# MiningSim Wireline Hole Inspection Workbench

<div style="padding:16px 18px;border:1px solid #d9e2ec;border-left:5px solid #1f6f78;border-radius:8px;background:#f7fafc;margin:10px 0 18px 0;">
<b>Purpose.</b> Prototype the ingestion, quality control, visualisation, uncertainty analysis, reconciliation and engineering calculations required for a MiningSim web application that inspects wireline logs from drilled blast holes.
</div>

This notebook reads the **actual LAS curve definitions** rather than assuming a fixed column order. It also reads GeoVista-style XHD metadata and calibration records from a user-selected folder, preserves XRD files for provenance, and provides reusable functions for:

- LAS/XHD folder inventory and validation;
- multi-track log inspection across gamma, caliper, temperature/conductivity and verticality tools;
- four-contact caliper calibration and direct tool-to-wall range inspection;
- uncertainty-aware cross-section and volume calculations using a family of feasible ellipses;
- borehole trajectory calculation from inclination and azimuth;
- repeated-run comparison and export-ready tables for a future web application.

> **Critical caliper interpretation.** `X1`, `X2`, `Y1` and `Y2` are treated as four tool-to-wall ranges measured from the tool's instantaneous, generally off-centre position. Opposite readings form two observed chords through the tool; they are **not borehole diameters**. Four contact points do not uniquely determine an ellipse, so the notebook no longer reports a single measured diameter, ovality, hole centre or volume. It reports direct observations, model-qualified lower bounds, an explicitly labelled reference scenario, and a finite upper bound only when defensible engineering constraints are supplied.

## 1. Scope, limitations and supplied-data findings

The supplied files use three companion formats:

- **LAS** — interpreted tabular log data and curve headers; this is the primary analytics input.
- **XHD** — XML metadata, sonde stack definitions, timestamps and calibration coefficients.
- **XRD** — proprietary binary acquisition data. It is inventoried and retained for traceability, but not decoded here because no public binary specification was supplied.

The notebook verifies these observations from the files themselves:

1. `Calibrated.las` and `Uncalibrated.las` contain four caliper channels: `X1`, `X2`, `Y1`, `Y2` in millimetres.
2. `Calibrated.xhd` contains four three-point polynomial calibrations using 150, 250 and 350 mm reference values.
3. `ACS-03_Run10_10m_min_up` has XHD/XRD companions but no LAS export.
4. The LAS `DATE` strings are malformed. They match `year-minute-day` from the XHD timestamp, so XHD `LogCreated` is treated as canonical.

### Caliper identifiability limitation

The four caliper channels are interpreted as ranges from a common tool reference to four wall contacts along two opposite, orthogonal tool axes. The tool reference is not assumed to coincide with the borehole centre, and the tool axes are not assumed to coincide with the principal axes of an elliptical hole. Therefore:

- `X1 + X2` and `Y1 + Y2` are **tool-axis chord lengths**, not diameters;
- a single ellipse, centre, ovality and cross-sectional area cannot be recovered from one four-arm station;
- under an exact ellipse model, the four contacts define a one-parameter family of feasible ellipses;
- that unconstrained family has no finite upper area or volume bound;
- any finite upper bound must be attributed to explicit engineering priors such as maximum axis ratio or maximum major diameter.

The notebook assumes the arm azimuths are 0°, 180°, 90° and 270° in tool coordinates. Confirm the physical arm geometry and measurement origin against manufacturer documentation before production deployment.

## 2. Imports, display theme and configuration

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable, Mapping, Optional, Sequence
import io
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

MS_COLOURS = ['#0B3C5D', '#1D70A2', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51', '#6C5B7B', '#59636E']

# ---------------------------------------------------------------------
# User input
# ---------------------------------------------------------------------
# Set this to the folder containing the extracted LAS, XHD and XRD files.
# Files may be located directly in this folder or inside nested subfolders.
INPUT_FOLDER = Path(r'./wireline_measurement_files').expanduser()
SUPPORTED_EXTENSIONS = {'.las', '.xhd', '.xrd'}

# ---------------------------------------------------------------------
# Hole-plan inputs — replace with the drill-and-blast plan/register.
# ---------------------------------------------------------------------
NOMINAL_DIAMETER_MM = 150.0
DIAMETER_TOLERANCE_MM = 10.0

# ---------------------------------------------------------------------
# Caliper acquisition/QC inputs
# ---------------------------------------------------------------------
CALIPER_MOTOR_CURRENT_MIN_MA = 1.8
CALIPER_MAX_RANGE_MM = 1_000.0
CALIPER_ARM_AZIMUTHS_DEG = {'X1': 0.0, 'X2': 180.0, 'Y1': 90.0, 'Y2': 270.0}

# ---------------------------------------------------------------------
# Ellipse-family calculation profile
# ---------------------------------------------------------------------
CALIPER_CALCULATION_PROFILE = 'four_contact_ellipse_family_v2'
CALIPER_ELLIPSE_RHO_SAMPLES = 401
CALIPER_ELLIPSE_RHO_MARGIN = 1e-4
CALIPER_FAMILY_DISPLAY_RHOS = (-0.85, -0.45, 0.0, 0.45, 0.85)

# The unconstrained ellipse family is unbounded above. A finite upper area
# or volume is produced only when a defensible size/shape constraint is set.
# Leave values as None until supported by tool geometry, centralisation,
# drilling practice, imaging, repeat runs or another independent source.
CALIPER_ELLIPSE_CONSTRAINTS = {
    'max_axis_ratio': None,                 # Example only: 2.0
    'max_major_diameter_mm': None,          # Example only: 300.0
    'max_tool_to_ellipse_centre_mm': None,  # Does not by itself guarantee a finite upper bound
}

TRAJECTORY_INCLINATION_REFERENCE = 'vertical'  # alternative: 'horizontal'
EXPORT_DIR = Path('./miningsim_exports')

HTML_STYLE = """
<style>
.jp-RenderedHTMLCommon table {font-size: 12px;}
.ms-card {display:inline-block;vertical-align:top;min-width:180px;margin:4px 8px 8px 0;padding:12px 14px;border:1px solid #d9e2ec;border-radius:8px;background:#ffffff;box-shadow:0 1px 2px rgba(0,0,0,.04)}
.ms-card .label {font-size:11px;text-transform:uppercase;letter-spacing:.04em;color:#59636e}
.ms-card .value {font-size:22px;font-weight:700;color:#0b3c5d;margin-top:4px}
.ms-callout {padding:13px 15px;border:1px solid #d9e2ec;border-left:5px solid #1f6f78;border-radius:8px;background:#f7fafc;margin:10px 0}
.ms-warning {padding:13px 15px;border:1px solid #f1d3a2;border-left:5px solid #e9a23b;border-radius:8px;background:#fffaf0;margin:10px 0}
.ms-ok {padding:13px 15px;border:1px solid #b7dfcf;border-left:5px solid #2a9d8f;border-radius:8px;background:#f3fbf8;margin:10px 0}
.ms-critical {padding:13px 15px;border:1px solid #efb6b2;border-left:5px solid #c74440;border-radius:8px;background:#fff6f5;margin:10px 0}
</style>
"""
display(HTML(HTML_STYLE))
print('Notebook environment ready.')

## 3. Data model and LAS/XHD parsers

In [ ]:
@dataclass
class CurveMeta:
    column: str
    mnemonic: str
    unit: str
    description: str
    ordinal: int
    sonde_name: Optional[str] = None
    sonde_serial: Optional[int] = None
    receiver_offset_m: Optional[float] = None


@dataclass
class CalibrationRecord:
    sonde_name: str
    sonde_serial: Optional[int]
    sonde_id: Optional[int]
    channel_name: str
    channel_index: int
    mode: str
    coefficients: tuple[float, float, float, float]
    raw_values: tuple[float, float, float, float]
    new_values: tuple[float, float, float, float]
    calibration_date: Optional[str]
    name_inferred: bool = False


@dataclass
class LogRecord:
    log_id: str
    archive_path: Path
    las_member: str
    xhd_member: Optional[str]
    xrd_member: Optional[str]
    df: pd.DataFrame
    curves: list[CurveMeta]
    version: dict[str, dict[str, str]]
    well: dict[str, dict[str, str]]
    parameters: dict[str, dict[str, str]]
    other: list[str]
    rejected_rows: int = 0
    xhd: dict[str, Any] = field(default_factory=dict)
    calibrations: list[CalibrationRecord] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)

    @property
    def index_unit(self) -> str:
        return self.curves[0].unit if self.curves else ''

    @property
    def view_type(self) -> Optional[str]:
        return self.xhd.get('view_type')

    @property
    def log_created(self) -> Optional[pd.Timestamp]:
        value = self.xhd.get('log_created')
        return pd.Timestamp(value) if value else None

    def curve(self, mnemonic: str, sonde_contains: Optional[str] = None, occurrence: int = 0) -> Optional[CurveMeta]:
        key = _normalise_name(mnemonic)
        matches = [curve for curve in self.curves if _normalise_name(curve.mnemonic) == key]
        if sonde_contains:
            sonde_key = sonde_contains.casefold()
            matches = [curve for curve in matches if curve.sonde_name and sonde_key in curve.sonde_name.casefold()]
        return matches[occurrence] if 0 <= occurrence < len(matches) else None


def _normalise_name(value: Any) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


def _parse_las_definition(line: str) -> Optional[dict[str, str]]:
    left, _, description = line.partition(':')
    match = re.match(r'^([^\.\s]+)\.([^\s]*)\s*(.*)$', left.strip())
    if not match:
        return None
    mnemonic, unit, value = match.groups()
    return {
        'mnemonic': mnemonic.strip(),
        'unit': unit.strip(),
        'value': value.strip(),
        'description': description.strip(),
    }


def parse_las_bytes(payload: bytes, source_name: str) -> dict[str, Any]:
    """Parse an unwrapped LAS 2.x file while preserving its declared curve order and units."""
    text = payload.decode('utf-8-sig', errors='replace')
    lines = text.splitlines()

    sections: dict[str, list[str]] = {}
    section_name = ''
    ascii_start: Optional[int] = None

    for index, raw_line in enumerate(lines):
        stripped = raw_line.strip()
        if stripped.startswith('~'):
            section_name = stripped.upper()
            if section_name.startswith('~ASCII'):
                ascii_start = index
                break
            sections.setdefault(section_name, [])
            continue
        if section_name:
            sections.setdefault(section_name, []).append(raw_line)

    if ascii_start is None:
        raise ValueError(f'{source_name}: LAS ASCII data section not found.')

    def section(prefix: str) -> list[str]:
        for key, values in sections.items():
            if key.startswith(prefix):
                return values
        return []

    def parse_named_section(prefix: str) -> dict[str, dict[str, str]]:
        output: dict[str, dict[str, str]] = {}
        for line in section(prefix):
            stripped = line.strip()
            if not stripped or stripped.startswith('#'):
                continue
            parsed = _parse_las_definition(stripped)
            if parsed:
                output[parsed['mnemonic'].upper()] = parsed
        return output

    version = parse_named_section('~VERSION')
    well = parse_named_section('~WELL')
    parameters = parse_named_section('~PARAMETER')

    wrap_value = version.get('WRAP', {}).get('value', 'NO').upper()
    if wrap_value not in {'NO', 'N'}:
        raise NotImplementedError(f'{source_name}: WRAP={wrap_value!r} is not supported by this prototype parser.')

    curve_definitions: list[dict[str, str]] = []
    for line in section('~CURVE'):
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        parsed = _parse_las_definition(stripped)
        if parsed:
            curve_definitions.append(parsed)

    if not curve_definitions:
        raise ValueError(f'{source_name}: no curve definitions were found.')

    occurrence_count: dict[str, int] = {}
    curves: list[CurveMeta] = []
    for ordinal, item in enumerate(curve_definitions):
        mnemonic = item['mnemonic']
        occurrence_count[mnemonic] = occurrence_count.get(mnemonic, 0) + 1
        suffix = occurrence_count[mnemonic]
        column = mnemonic if suffix == 1 else f'{mnemonic}__{suffix}'
        curves.append(CurveMeta(
            column=column,
            mnemonic=mnemonic,
            unit=item['unit'],
            description=item['description'],
            ordinal=ordinal,
        ))

    rows: list[np.ndarray] = []
    rejected_rows = 0
    expected_columns = len(curves)
    for line in lines[ascii_start + 1:]:
        stripped = line.strip()
        if not stripped or stripped.startswith('#'):
            continue
        values = np.fromstring(stripped, sep=' ')
        if values.size != expected_columns:
            rejected_rows += 1
            continue
        rows.append(values)

    if not rows:
        raise ValueError(f'{source_name}: no valid numeric rows were found.')

    frame = pd.DataFrame(np.vstack(rows), columns=[curve.column for curve in curves])
    null_text = well.get('NULL', {}).get('value', '-999.25').split()[0]
    try:
        null_value = float(null_text)
        frame = frame.mask(np.isclose(frame, null_value, equal_nan=False))
    except ValueError:
        null_value = np.nan

    other = [
        line.strip() for line in section('~OTHER')
        if line.strip() and not line.strip().startswith('#')
    ]

    return {
        'df': frame,
        'curves': curves,
        'version': version,
        'well': well,
        'parameters': parameters,
        'other': other,
        'rejected_rows': rejected_rows,
        'null_value': null_value,
    }

In [ ]:
def _safe_int(value: Optional[str]) -> Optional[int]:
    try:
        return int(value) if value not in (None, '') else None
    except (TypeError, ValueError):
        return None


def _safe_float(value: Optional[str]) -> Optional[float]:
    try:
        return float(value) if value not in (None, '') else None
    except (TypeError, ValueError):
        return None


def _float_tuple(parent: ET.Element, path: str) -> tuple[float, float, float, float]:
    values = [_safe_float(node.text) or 0.0 for node in parent.findall(path)]
    values = (values + [0.0, 0.0, 0.0, 0.0])[:4]
    return tuple(values)  # type: ignore[return-value]


def parse_xhd_bytes(payload: bytes) -> dict[str, Any]:
    """Parse the GeoVista XML header and calibration records."""
    root = ET.fromstring(payload.decode('utf-8-sig', errors='replace'))
    nil_attr = '{http://www.w3.org/2001/XMLSchema-instance}nil'

    sonde_definitions: dict[int, dict[str, Any]] = {}
    for definition in root.findall('./Sondes/SondeDefinition'):
        sonde_id = _safe_int(definition.findtext('SondeID'))
        if sonde_id is None:
            continue
        channels = []
        for channel in definition.findall('.//ChannelList/SondeChannel'):
            channels.append({
                'channel_name': channel.findtext('ChannelName') or '',
                'channel_id': _safe_int(channel.findtext('ChannelID')),
                'units': channel.findtext('Units') or '',
                'receiver_offset_m': _safe_float(channel.findtext('ReceiverOffset')),
            })
        sonde_definitions[sonde_id] = {
            'sonde_id': sonde_id,
            'sonde_name': definition.findtext('SondeName') or f'Sonde {sonde_id}',
            'channels': channels,
        }

    selected_ids = [_safe_int(node.text) or 0 for node in root.findall('./SelectedStackDefinition/SondeID/int')]
    selected_serials = [_safe_int(node.text) or 0 for node in root.findall('./SelectedStackDefinition/SondeSerial/int')]
    selected_enabled = [(node.text or '').strip().casefold() == 'true' for node in root.findall('./SelectedStackDefinition/IsEnabled/boolean')]
    selected_stack = []
    for index, sonde_id in enumerate(selected_ids):
        if sonde_id == 0:
            continue
        enabled = selected_enabled[index] if index < len(selected_enabled) else True
        if not enabled:
            continue
        definition = sonde_definitions.get(sonde_id, {'sonde_name': f'Sonde {sonde_id}', 'channels': []})
        selected_stack.append({
            'sonde_id': sonde_id,
            'sonde_serial': selected_serials[index] if index < len(selected_serials) else None,
            'sonde_name': definition['sonde_name'],
            'channels': definition['channels'],
        })

    calibrations: list[CalibrationRecord] = []
    for sonde_cal in root.findall('./Calibration/SondeCalibration'):
        if sonde_cal.get(nil_attr) == 'true':
            continue
        sonde_name = sonde_cal.findtext('SondeName') or ''
        sonde_serial = _safe_int(sonde_cal.findtext('SondeSerial'))
        sonde_id = _safe_int(sonde_cal.findtext('SondeID'))
        calibration_date = sonde_cal.findtext('CalibrationDate')
        channel_definitions = sonde_definitions.get(sonde_id or -1, {}).get('channels', [])

        for channel_index, calibration in enumerate(sonde_cal.findall('.//CalibrationData')):
            explicit_name = (calibration.findtext('ChannelName') or '').strip()
            inferred_name = ''
            if not explicit_name and channel_index < len(channel_definitions):
                inferred_name = channel_definitions[channel_index].get('channel_name', '')
            channel_name = explicit_name or inferred_name or f'channel_{channel_index}'
            calibrations.append(CalibrationRecord(
                sonde_name=sonde_name,
                sonde_serial=sonde_serial,
                sonde_id=sonde_id,
                channel_name=channel_name,
                channel_index=channel_index,
                mode=calibration.findtext('Mode') or 'None',
                coefficients=_float_tuple(calibration, './Coefficients/decimal'),
                raw_values=_float_tuple(calibration, './RawDataValues/decimal'),
                new_values=_float_tuple(calibration, './NewValues/decimal'),
                calibration_date=calibration_date,
                name_inferred=not bool(explicit_name) and bool(inferred_name),
            ))

    return {
        'software_version': root.findtext('SoftwareVersion'),
        'log_created': root.findtext('LogCreated'),
        'view_type': root.findtext('ViewType'),
        'stack_length_m': _safe_float(root.findtext('StackLength')),
        'selected_stack': selected_stack,
        'sonde_definitions': sonde_definitions,
        'calibrations': calibrations,
        'general_view': {
            'start_depth_m': _safe_float(root.findtext('./GeneralView/StartDepth')),
            'end_depth_m': _safe_float(root.findtext('./GeneralView/EndDepth')),
            'depth_when_stopped_m': _safe_float(root.findtext('./GeneralView/DepthWhenLogStopped')),
        },
    }


def assign_curve_owners(curves: list[CurveMeta], xhd: Mapping[str, Any]) -> list[CurveMeta]:
    """Attach sonde ownership by matching the reversed selected stack to the LAS curve sequence."""
    output = [replace(curve) for curve in curves]
    current_positions = [i for i, curve in enumerate(output) if _normalise_name(curve.mnemonic) == 'current']
    start = current_positions[0] + 1 if current_positions else 0

    flattened: list[dict[str, Any]] = []
    for sonde in reversed(list(xhd.get('selected_stack', []))):
        for channel in sonde.get('channels', []):
            flattened.append({**channel, **{
                'sonde_name': sonde.get('sonde_name'),
                'sonde_serial': sonde.get('sonde_serial'),
            }})

    for curve, channel in zip(output[start:], flattened):
        if _normalise_name(curve.mnemonic) != _normalise_name(channel.get('channel_name', '')):
            # Do not force a potentially incorrect ownership assignment.
            continue
        curve.sonde_name = channel.get('sonde_name')
        curve.sonde_serial = channel.get('sonde_serial')
        curve.receiver_offset_m = channel.get('receiver_offset_m')
        if not curve.unit and channel.get('units'):
            curve.unit = channel['units']
    return output

## 4. Discover and load measurement files from a user-defined folder

In [ ]:
def discover_measurement_files(input_folder: Path) -> list[Path]:
    """Recursively discover supported wireline measurement files."""
    input_folder = Path(input_folder).expanduser().resolve()

    if not input_folder.exists():
        raise FileNotFoundError(
            f"Input folder does not exist: {input_folder}\n"
            "Update INPUT_FOLDER in the configuration cell."
        )
    if not input_folder.is_dir():
        raise NotADirectoryError(f"Input path is not a folder: {input_folder}")

    files = [
        path.resolve()
        for path in input_folder.rglob('*')
        if path.is_file() and path.suffix.casefold() in SUPPORTED_EXTENSIONS
    ]
    if not files:
        raise FileNotFoundError(
            f"No LAS, XHD or XRD files were found in: {input_folder}"
        )

    return sorted(
        files,
        key=lambda path: str(path.relative_to(input_folder)).casefold(),
    )


def load_measurement_folder(
    input_folder: Path,
) -> tuple[dict[str, LogRecord], pd.DataFrame]:
    """
    Load LAS files and matching XHD/XRD companions from a folder.

    Companion files are matched case-insensitively by their parent folder,
    filename stem and extension. Nested subfolders are supported.
    """
    input_folder = Path(input_folder).expanduser().resolve()
    discovered_files = discover_measurement_files(input_folder)

    files_by_stem: dict[tuple[Path, str], dict[str, Path]] = {}
    for path in discovered_files:
        key = (path.parent, path.stem.casefold())
        files_by_stem.setdefault(key, {})[path.suffix.casefold()] = path

    logs: dict[str, LogRecord] = {}
    inventory_rows: list[dict[str, Any]] = []

    for (parent_folder, _), companion_files in sorted(
        files_by_stem.items(),
        key=lambda item: str(item[0][0] / item[0][1]).casefold(),
    ):
        representative_path = next(iter(companion_files.values()))
        item_name = representative_path.stem
        las_path = companion_files.get('.las')
        xhd_path = companion_files.get('.xhd')
        xrd_path = companion_files.get('.xrd')

        relative_folder = parent_folder.relative_to(input_folder)
        relative_folder_text = '' if str(relative_folder) == '.' else str(relative_folder)

        inventory_rows.append({
            'source_folder': relative_folder_text,
            'item': item_name,
            'LAS': las_path is not None,
            'XHD': xhd_path is not None,
            'XRD': xrd_path is not None,
            'complete_triplet': all(path is not None for path in (las_path, xhd_path, xrd_path)),
        })

        # XHD/XRD-only records remain visible in the inventory but cannot be
        # parsed as logs without a LAS file.
        if las_path is None:
            continue

        relative_las_path = las_path.relative_to(input_folder)
        parsed = parse_las_bytes(
            las_path.read_bytes(),
            str(relative_las_path).replace('\\', '/'),
        )
        xhd = parse_xhd_bytes(xhd_path.read_bytes()) if xhd_path else {}
        curves = assign_curve_owners(parsed['curves'], xhd) if xhd else parsed['curves']

        # Relative paths provide stable, unique IDs when different subfolders
        # contain files with the same basename.
        log_id = str(relative_las_path.with_suffix('')).replace('\\', '/')
        unique_id = log_id
        duplicate_index = 2
        while unique_id in logs:
            unique_id = f'{log_id}::{duplicate_index}'
            duplicate_index += 1

        warnings_list: list[str] = []
        raw_las_date = parsed['well'].get('DATE', {}).get('value')
        log_created = xhd.get('log_created')
        if raw_las_date and log_created:
            try:
                stamp = pd.Timestamp(log_created)
                logger_bug_value = f'{stamp.year:04d}-{stamp.minute:02d}-{stamp.day:02d}'
                if raw_las_date == logger_bug_value:
                    warnings_list.append(
                        'LAS DATE encodes year-minute-day; XHD LogCreated is canonical.'
                    )
            except Exception:
                pass

        logs[unique_id] = LogRecord(
            log_id=unique_id,
            archive_path=input_folder,  # retained for backward compatibility
            las_member=str(relative_las_path).replace('\\', '/'),
            xhd_member=(
                str(xhd_path.relative_to(input_folder)).replace('\\', '/')
                if xhd_path else None
            ),
            xrd_member=(
                str(xrd_path.relative_to(input_folder)).replace('\\', '/')
                if xrd_path else None
            ),
            df=parsed['df'],
            curves=curves,
            version=parsed['version'],
            well=parsed['well'],
            parameters=parsed['parameters'],
            other=parsed['other'],
            rejected_rows=parsed['rejected_rows'],
            xhd=xhd,
            calibrations=list(xhd.get('calibrations', [])),
            warnings=warnings_list,
        )

    inventory = pd.DataFrame(inventory_rows)
    if not inventory.empty:
        inventory = inventory.sort_values(
            ['source_folder', 'item']
        ).reset_index(drop=True)

    return logs, inventory


def classify_log(log: LogRecord) -> str:
    """Classify a log from the measurement curves it contains."""
    names = {_normalise_name(curve.mnemonic) for curve in log.curves}
    tags: list[str] = []

    if {'x1', 'x2', 'y1', 'y2'}.issubset(names):
        tags.append('4-arm caliper')
    if 'gr' in names or 'api' in names:
        tags.append('gamma')
    if {'incline', 'azimuth'}.issubset(names):
        tags.append('directional')
    if 'conductivity' in names or 'pt100' in names:
        tags.append('temperature/conductivity')
    if log.index_unit.casefold().startswith('second') or (log.view_type or '').casefold() == 'time':
        tags.insert(0, 'telemetry/time-series')

    return ' + '.join(dict.fromkeys(tags)) or 'generic LAS'


def physical_depth_column(log: LogRecord) -> str:
    """Return the physical depth or time-index dataframe column."""
    if log.df.empty:
        raise ValueError(f'Log {log.log_id!r} contains no data.')
    if not log.curves:
        raise ValueError(f'Log {log.log_id!r} contains no curve metadata.')
    if log.index_unit.casefold().startswith('second'):
        return log.curves[0].column

    candidate = log.curve('Depth')
    if (
        candidate is not None
        and candidate.column in log.df.columns
        and log.df[candidate.column].notna().sum() >= 2
    ):
        return candidate.column

    normalised_columns = {
        _normalise_name(column): column for column in log.df.columns
    }
    for name in ('dept', 'depth', 'md', 'measureddepth'):
        if name in normalised_columns:
            return normalised_columns[name]

    # LAS convention places the index curve first.
    return log.curves[0].column


def curve_unit(log: LogRecord, column: str) -> str:
    """Return the declared LAS unit for a dataframe column."""
    match = next((curve for curve in log.curves if curve.column == column), None)
    return match.unit if match else ''


MEASUREMENT_FILES = discover_measurement_files(INPUT_FOLDER)
print(f'Input folder: {INPUT_FOLDER.resolve()}')
print(f'Discovered {len(MEASUREMENT_FILES)} supported files:')
for path in MEASUREMENT_FILES:
    print(f'  • {path.relative_to(INPUT_FOLDER.resolve())}')

logs, archive_inventory = load_measurement_folder(INPUT_FOLDER)
if not logs:
    raise FileNotFoundError(
        'Measurement files were found, but no LAS files could be loaded.'
    )

print(f'\nLoaded {len(logs)} LAS logs from the selected folder.')

In [ ]:
def log_summary_table(logs: Mapping[str, LogRecord]) -> pd.DataFrame:
    rows = []
    for log_id, log in logs.items():
        depth_col = physical_depth_column(log)
        depth = pd.to_numeric(log.df[depth_col], errors='coerce').dropna()
        raw_date = log.well.get('DATE', {}).get('value')
        median_step = float(np.nanmedian(np.abs(np.diff(depth)))) if len(depth) > 1 else np.nan
        duplicate_mnemonics = pd.Series([curve.mnemonic for curve in log.curves]).duplicated().sum()
        rows.append({
            'log_id': log_id,
            'source_folder': str(Path(log.las_member).parent) if str(Path(log.las_member).parent) != '.' else '',
            'classification': classify_log(log),
            'view': log.view_type,
            'created_from_xhd': str(log.log_created) if log.log_created is not None else None,
            'LAS_DATE_raw': raw_date,
            'rows': len(log.df),
            'curves': len(log.curves),
            'index_unit': log.index_unit,
            'depth_min': float(depth.min()) if not depth.empty else np.nan,
            'depth_max': float(depth.max()) if not depth.empty else np.nan,
            'median_sample_step': median_step,
            'duplicate_mnemonics': int(duplicate_mnemonics),
            'active_calibrations': sum(record.mode != 'None' for record in log.calibrations),
            'rejected_rows': log.rejected_rows,
            'warnings': ' | '.join(log.warnings),
        })
    return pd.DataFrame(rows).sort_values(['source_folder', 'log_id']).reset_index(drop=True)

summary_table = log_summary_table(logs)
display(summary_table)

incomplete = archive_inventory.loc[~archive_inventory['complete_triplet']].copy()
if not incomplete.empty:
    display(HTML('<div class="ms-warning"><b>Incomplete companion sets found.</b> The items below cannot be fully processed as LAS logs.</div>'))
    display(incomplete)
else:
    display(HTML('<div class="ms-ok"><b>All discovered items contain LAS, XHD and XRD companions.</b></div>'))

## 5. Confirm the caliper content and calibration records

In [ ]:
ARM_NAMES = ('X1', 'X2', 'Y1', 'Y2')


def is_caliper_log(log: LogRecord) -> bool:
    return all(log.curve(name) is not None for name in ARM_NAMES)


def caliper_calibration_table(log: LogRecord) -> pd.DataFrame:
    rows = []
    for record in log.calibrations:
        if 'caliper' not in record.sonde_name.casefold() or record.channel_name not in ARM_NAMES:
            continue
        rows.append({
            'channel': record.channel_name,
            'mode': record.mode,
            'c0': record.coefficients[0],
            'c1': record.coefficients[1],
            'c2': record.coefficients[2],
            'c3': record.coefficients[3],
            'raw_point_1': record.raw_values[0],
            'raw_point_2': record.raw_values[1],
            'raw_point_3': record.raw_values[2],
            'reference_mm_1': record.new_values[0],
            'reference_mm_2': record.new_values[1],
            'reference_mm_3': record.new_values[2],
            'calibration_date': record.calibration_date,
            'channel_name_inferred_from_order': record.name_inferred,
        })
    return pd.DataFrame(rows)

caliper_ids = [log_id for log_id, log in logs.items() if is_caliper_log(log)]
if not caliper_ids:
    raise AssertionError('No four-arm caliper log was detected.')

print('Caliper logs:', caliper_ids)
for log_id in caliper_ids:
    log = logs[log_id]
    curves = [curve for curve in log.curves if curve.mnemonic in ARM_NAMES]
    display(pd.DataFrame([{
        'log_id': log_id,
        'column': curve.column,
        'mnemonic': curve.mnemonic,
        'unit': curve.unit,
        'sonde': curve.sonde_name,
        'serial': curve.sonde_serial,
        'receiver_offset_m': curve.receiver_offset_m,
        'non_null_samples': int(log.df[curve.column].notna().sum()),
        'median': float(log.df[curve.column].median()),
        'maximum': float(log.df[curve.column].max()),
    } for curve in curves]))

calibrated_id = next((item for item in caliper_ids if Path(item).name.casefold() == 'calibrated'), caliper_ids[0])
uncalibrated_id = next((item for item in caliper_ids if 'uncalibrated' in Path(item).name.casefold()), None)
calibrated_log = logs[calibrated_id]
uncalibrated_log = logs[uncalibrated_id] if uncalibrated_id else None

cal_table = caliper_calibration_table(calibrated_log)
display(HTML('<div class="ms-ok"><b>Four caliper curves found.</b> The calibrated XHD also contains the following polynomial records.</div>'))
display(cal_table)

### Interpretation of the two caliper LAS files

`Calibrated.las` contains values on a physical millimetre scale. `Uncalibrated.las` contains raw counts in the millions. The calibration coefficients are stored only in `Calibrated.xhd`; their channel names are blank in the XML, so the parser maps them to `X1`, `X2`, `Y1`, `Y2` by the declared caliper channel order and marks that mapping as inferred.

The four calibrated channels are treated as **tool-to-wall ranges**, not radii from the borehole centre. In tool coordinates the measured contacts are represented as:

- `X1`: `( +X1, 0 )`
- `X2`: `( -X2, 0 )`
- `Y1`: `( 0, +Y1 )`
- `Y2`: `( 0, -Y2 )`

This coordinate representation requires the X arms to be opposite, the Y arms to be opposite, and the two pairs to be orthogonal. It does not require the tool to be centred or its axes to align with the hole's principal axes. If the physical arm azimuths or range origins differ from this convention, update the model before using the derived uncertainty calculations.

## 6. Quality-control functions and curve audit

In [ ]:
def curve_audit(log: LogRecord) -> pd.DataFrame:
    rows = []
    for curve in log.curves:
        series = pd.to_numeric(log.df[curve.column], errors='coerce')
        valid = series.replace([np.inf, -np.inf], np.nan).dropna()
        flag = ''
        name = _normalise_name(curve.mnemonic)
        if not valid.empty:
            if name == 'temperature' and ((valid < -100).any() or (valid > 250).any()):
                flag = 'outside typical physical temperature range; may be raw/invalid'
            elif name == 'incline' and ((valid < 0).any() or (valid > 180).any()):
                flag = 'outside 0–180°'
            elif name == 'azimuth' and ((valid < 0).any() or (valid > 360).any()):
                flag = 'outside 0–360°'
            elif name in {'x1', 'x2', 'y1', 'y2'} and valid.median() > 10_000:
                flag = 'raw counts rather than calibrated millimetres'
        rows.append({
            'column': curve.column,
            'mnemonic': curve.mnemonic,
            'sonde': curve.sonde_name,
            'serial': curve.sonde_serial,
            'unit': curve.unit,
            'receiver_offset_m': curve.receiver_offset_m,
            'valid_samples': int(valid.size),
            'missing_pct': 100.0 * (1.0 - valid.size / max(len(series), 1)),
            'minimum': float(valid.min()) if not valid.empty else np.nan,
            'median': float(valid.median()) if not valid.empty else np.nan,
            'maximum': float(valid.max()) if not valid.empty else np.nan,
            'QC_flag': flag,
        })
    return pd.DataFrame(rows)


def log_qc_issues(log: LogRecord) -> list[str]:
    issues = list(log.warnings)
    if log.rejected_rows:
        issues.append(f'{log.rejected_rows} ASCII rows did not match the declared curve count.')
    depth_col = physical_depth_column(log)
    depth = pd.to_numeric(log.df[depth_col], errors='coerce').dropna()
    if len(depth) > 2:
        diffs = np.diff(depth)
        direction = np.sign(np.nanmedian(diffs))
        reversals = int(np.sum(np.sign(diffs[np.abs(diffs) > 1e-9]) != direction))
        if reversals:
            issues.append(f'{reversals} depth-direction reversals detected.')
    audit = curve_audit(log)
    flagged = audit.loc[audit['QC_flag'].ne(''), ['column', 'QC_flag']]
    for row in flagged.itertuples(index=False):
        issues.append(f'{row.column}: {row.QC_flag}.')
    return issues

qc_rows = []
for log_id, log in logs.items():
    issues = log_qc_issues(log)
    qc_rows.append({'log_id': log_id, 'issue_count': len(issues), 'issues': ' | '.join(issues)})
qc_summary = pd.DataFrame(qc_rows).sort_values(['issue_count', 'log_id'], ascending=[False, True])
display(qc_summary)

# Inspect the most complete directional run and the two caliper logs in detail.
for audit_id in [item for item in ['ACS-03_Run10_10m_min_down', calibrated_id, uncalibrated_id] if item and item in logs]:
    display(Markdown(f'**Curve audit — `{audit_id}`**'))
    display(curve_audit(logs[audit_id]))

## 7. Reusable multi-track log viewer

In [ ]:
def resolve_track(log: LogRecord, track: str) -> Optional[str]:
    if track in log.df.columns:
        return track
    match = log.curve(track)
    return match.column if match else None


def plot_log_tracks(
    log: LogRecord,
    tracks: Sequence[str],
    title: Optional[str] = None,
    depth_column: Optional[str] = None,
    reverse_depth: bool = True,
    height: int = 760,
) -> go.Figure:
    depth_column = depth_column or physical_depth_column(log)
    resolved = [resolve_track(log, track) for track in tracks]
    resolved = [track for track in resolved if track is not None]
    if not resolved:
        raise ValueError('None of the requested tracks is available in this log.')

    fig = make_subplots(rows=1, cols=len(resolved), shared_yaxes=True, horizontal_spacing=0.035)
    for index, column in enumerate(resolved, start=1):
        temp = log.df[[depth_column, column]].apply(pd.to_numeric, errors='coerce').dropna()
        fig.add_trace(go.Scatter(
            x=temp[column], y=temp[depth_column], mode='lines',
            name=column, line={'width': 1.6, 'color': MS_COLOURS[(index - 1) % len(MS_COLOURS)]},
            hovertemplate=f'{column}: %{{x:.4g}}<br>Depth: %{{y:.3f}}<extra></extra>',
        ), row=1, col=index)
        unit = curve_unit(log, column)
        fig.update_xaxes(title_text=f'{column}<br><span style="font-size:10px">{unit}</span>', row=1, col=index)

    y_title = f'{depth_column} ({curve_unit(log, depth_column) or log.index_unit})'
    fig.update_yaxes(title_text=y_title, autorange='reversed' if reverse_depth else True, row=1, col=1)
    fig.update_layout(
        title=title or f'{log.log_id} — wireline tracks',
        template='plotly_white', height=height,
        width=max(900, 235 * len(resolved)),
        showlegend=False, hovermode='y unified',
        margin={'l': 70, 'r': 30, 't': 80, 'b': 60},
    )
    return fig


def find_curve_column(log: LogRecord, mnemonic: str, sonde_contains: Optional[str] = None) -> Optional[str]:
    curve = log.curve(mnemonic, sonde_contains=sonde_contains)
    return curve.column if curve else None

# Run 10 is the supplied ACS log with non-zero verticality channels and physically scaled T/C data.
directional_example_id = next((key for key in logs if 'Run10_10m_min_down' in key), None)
if directional_example_id:
    directional_example = logs[directional_example_id]
    example_tracks = [
        find_curve_column(directional_example, 'GR', 'Gamma'),
        find_curve_column(directional_example, 'Conductivity', 'Conductivity'),
        find_curve_column(directional_example, 'Temperature', 'Conductivity'),
        find_curve_column(directional_example, 'Incline', 'Verticality'),
        find_curve_column(directional_example, 'Azimuth', 'Verticality'),
    ]
    example_tracks = [track for track in example_tracks if track]
    plot_log_tracks(directional_example, example_tracks, title=f'{directional_example_id} — combined tool string').show()

## 8. Caliper calibration and calibrated/uncalibrated comparison

In [ ]:
def polynomial_calibration(values: pd.Series, coefficients: Sequence[float]) -> pd.Series:
    c0, c1, c2, c3 = list(coefficients)[:4]
    numeric = pd.to_numeric(values, errors='coerce')
    return c0 + c1 * numeric + c2 * numeric.pow(2) + c3 * numeric.pow(3)


def caliper_looks_raw(log: LogRecord) -> bool:
    medians = []
    for arm in ARM_NAMES:
        curve = log.curve(arm)
        if curve:
            medians.append(float(log.df[curve.column].median()))
    return bool(medians) and float(np.nanmedian(medians)) > 10_000


def calibrated_caliper_frame(raw_log: LogRecord, calibration_source: LogRecord) -> pd.DataFrame:
    output = raw_log.df.copy()
    records = {
        record.channel_name: record
        for record in calibration_source.calibrations
        if 'caliper' in record.sonde_name.casefold()
        and record.channel_name in ARM_NAMES
        and record.mode != 'None'
    }
    missing = [arm for arm in ARM_NAMES if arm not in records]
    if missing:
        raise ValueError(f'Calibration source is missing records for: {missing}')
    for arm in ARM_NAMES:
        source_curve = raw_log.curve(arm)
        if source_curve is None:
            raise ValueError(f'Raw log is missing {arm}.')
        output[source_curve.column] = polynomial_calibration(output[source_curve.column], records[arm].coefficients)
    return output

print(f'{calibrated_id}: raw-count heuristic = {caliper_looks_raw(calibrated_log)}')
if uncalibrated_log is not None:
    print(f'{uncalibrated_id}: raw-count heuristic = {caliper_looks_raw(uncalibrated_log)}')
    uncalibrated_scaled = calibrated_caliper_frame(uncalibrated_log, calibrated_log)

    comparison_rows = []
    for arm in ARM_NAMES:
        raw_col = uncalibrated_log.curve(arm).column
        comparison_rows.append({
            'arm': arm,
            'raw_median': float(uncalibrated_log.df[raw_col].median()),
            'scaled_median_mm': float(uncalibrated_scaled[raw_col].median()),
            'calibrated_LAS_median_mm': float(calibrated_log.df[calibrated_log.curve(arm).column].median()),
        })
    display(pd.DataFrame(comparison_rows))


def compare_caliper_runs(
    physical_log: LogRecord,
    raw_log: LogRecord,
    raw_scaled_frame: pd.DataFrame,
    step_m: float = 0.02,
) -> pd.DataFrame:
    depth_a = physical_depth_column(physical_log)
    depth_b = physical_depth_column(raw_log)
    a = pd.DataFrame({'depth': physical_log.df[depth_a]})
    b = pd.DataFrame({'depth': raw_log.df[depth_b]})
    for arm in ARM_NAMES:
        a[arm] = physical_log.df[physical_log.curve(arm).column]
        b[arm] = raw_scaled_frame[raw_log.curve(arm).column]
    a = a.dropna().groupby('depth', as_index=False).median().sort_values('depth')
    b = b.dropna().groupby('depth', as_index=False).median().sort_values('depth')
    lower = max(a['depth'].min(), b['depth'].min())
    upper = min(a['depth'].max(), b['depth'].max())
    grid = np.arange(lower, upper + step_m / 2, step_m)
    rows = []
    for arm in ARM_NAMES:
        av = np.interp(grid, a['depth'], a[arm])
        bv = np.interp(grid, b['depth'], b[arm])
        rows.append({
            'arm': arm,
            'samples': len(grid),
            'correlation': float(np.corrcoef(av, bv)[0, 1]),
            'mean_bias_scaled_minus_calibrated_mm': float(np.mean(bv - av)),
            'MAE_mm': float(np.mean(np.abs(bv - av))),
            'RMSE_mm': float(np.sqrt(np.mean((bv - av) ** 2))),
        })
    return pd.DataFrame(rows)

if uncalibrated_log is not None:
    display(HTML('<div class="ms-callout"><b>Calibration cross-check.</b> These are separate logging runs, so residuals include repeatability, depth alignment and tool-position differences; they are not a pure calibration error.</div>'))
    display(compare_caliper_runs(calibrated_log, uncalibrated_log, uncalibrated_scaled))

## 9. Four-contact caliper uncertainty, volume bounds and reconciliation

The previous notebook added opposite readings and treated the results as orthogonal diameters. That is not supported by the acquisition geometry. The sums remain useful, but only as the lengths of two chords passing through the instantaneous tool reference.

For positive ranges `X1`, `X2`, `Y1`, `Y2`, define the four contact points in tool coordinates as `(X1,0)`, `(-X2,0)`, `(0,Y1)` and `(0,-Y2)`. Every exact ellipse through those four points can be written as:

\[
q_x x^2 + 2\rho\sqrt{q_x q_y}\,xy + q_y y^2 + \ell_x x + \ell_y y - 1 = 0,\qquad -1 < \rho < 1,
\]

where

\[
q_x=\frac{1}{X1\,X2},\quad q_y=\frac{1}{Y1\,Y2},\quad
\ell_x=\frac{X2-X1}{X1\,X2},\quad
\ell_y=\frac{Y2-Y1}{Y1\,Y2}.
\]

The free parameter `ρ` changes ellipse rotation, centre, axes and area while retaining all four measured contacts. As `|ρ|` approaches 1, the quadratic form approaches singularity and ellipse area can grow without limit. Consequently, four contacts alone cannot provide a finite upper volume.

This section reports four distinct products:

1. **Contact-polygon area** — the area of the quadrilateral joining the four measured contacts. It is a model-free geometric quantity and a lower bound on cross-sectional area only if the true section is convex and contains the contact polygon.
2. **Minimum feasible ellipse** — the minimum-area member of the exact ellipse family. It is a lower bound only within the exact, convex ellipse model.
3. **Axis-aligned reference ellipse (`ρ = 0`)** — a reproducible scenario whose principal axes align with the tool axes. It is not a measurement or a best estimate.
4. **Constrained interval** — a lower and upper ellipse area obtained only after applying explicit engineering constraints. Without a maximum axis ratio or maximum major diameter, the upper bound remains unbounded.

No output in this section should be labelled “measured diameter”, “measured ovality”, “inferred hole centre” or “measured volume”.

In [ ]:
def _normalise_ellipse_constraints(constraints: Optional[Mapping[str, Optional[float]]]) -> dict[str, Optional[float]]:
    output = {
        'max_axis_ratio': None,
        'max_major_diameter_mm': None,
        'max_tool_to_ellipse_centre_mm': None,
    }
    if constraints:
        unknown = set(constraints) - set(output)
        if unknown:
            raise ValueError(f'Unknown ellipse constraints: {sorted(unknown)}')
        output.update(constraints)
    for name, value in output.items():
        if value is not None:
            value = float(value)
            if not np.isfinite(value) or value <= 0:
                raise ValueError(f'{name} must be a positive finite number or None.')
            output[name] = value
    if output['max_axis_ratio'] is not None and output['max_axis_ratio'] < 1.0:
        raise ValueError('max_axis_ratio must be at least 1.0.')
    return output


def ellipse_constraints_label(constraints: Mapping[str, Optional[float]]) -> str:
    active = [f'{name}={value:g}' for name, value in constraints.items() if value is not None]
    return '; '.join(active) if active else 'none — upper bound unbounded'


def make_ellipse_rho_grid(samples: int = 401, margin: float = 1e-4) -> np.ndarray:
    """Return a deterministic grid with extra resolution near the singular limits ±1."""
    if samples < 101:
        raise ValueError('samples must be at least 101.')
    if not 0 < margin < 0.1:
        raise ValueError('margin must lie between 0 and 0.1.')
    linear = np.linspace(-1.0 + margin, 1.0 - margin, samples)
    edge_limit = np.arctanh(1.0 - margin)
    edge_dense = np.tanh(np.linspace(-edge_limit, edge_limit, samples))
    return np.unique(np.concatenate([linear, edge_dense, np.array([0.0])]))


def ellipse_family_arrays(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    rho_values: np.ndarray,
) -> dict[str, np.ndarray]:
    """Evaluate the exact one-parameter ellipse family through four orthogonal contacts."""
    ranges = np.asarray([x1_mm, x2_mm, y1_mm, y2_mm], dtype=float)
    if not np.isfinite(ranges).all() or np.any(ranges <= 0):
        raise ValueError('All four caliper ranges must be positive and finite.')

    rho = np.asarray(rho_values, dtype=float)
    if np.any(np.abs(rho) >= 1.0):
        raise ValueError('Every rho value must satisfy -1 < rho < 1.')

    qxx = 1.0 / (x1_mm * x2_mm)
    qyy = 1.0 / (y1_mm * y2_mm)
    qxy = rho * math.sqrt(qxx * qyy)
    determinant = qxx * qyy - qxy ** 2

    linear_x = (x2_mm - x1_mm) * qxx
    linear_y = (y2_mm - y1_mm) * qyy
    inverse_times_linear_x = (qyy * linear_x - qxy * linear_y) / determinant
    inverse_times_linear_y = (-qxy * linear_x + qxx * linear_y) / determinant

    centre_x_mm = -0.5 * inverse_times_linear_x
    centre_y_mm = -0.5 * inverse_times_linear_y
    level = 1.0 + 0.25 * (
        linear_x * inverse_times_linear_x
        + linear_y * inverse_times_linear_y
    )

    trace = qxx + qyy
    discriminant = np.sqrt((qxx - qyy) ** 2 + 4.0 * qxy ** 2)
    eigenvalue_min = (trace - discriminant) / 2.0
    eigenvalue_max = (trace + discriminant) / 2.0

    semi_major_mm = np.sqrt(level / eigenvalue_min)
    semi_minor_mm = np.sqrt(level / eigenvalue_max)
    area_mm2 = math.pi * level / np.sqrt(determinant)

    return {
        'rho': rho,
        'area_mm2': area_mm2,
        'area_equivalent_diameter_mm': 2.0 * np.sqrt(area_mm2 / math.pi),
        'centre_x_mm': centre_x_mm,
        'centre_y_mm': centre_y_mm,
        'tool_to_ellipse_centre_mm': np.hypot(centre_x_mm, centre_y_mm),
        'semi_major_mm': semi_major_mm,
        'semi_minor_mm': semi_minor_mm,
        'major_diameter_mm': 2.0 * semi_major_mm,
        'minor_diameter_mm': 2.0 * semi_minor_mm,
        'axis_ratio': semi_major_mm / semi_minor_mm,
        'level': level,
    }


def ellipse_family_table(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    rho_values: np.ndarray,
) -> pd.DataFrame:
    return pd.DataFrame(ellipse_family_arrays(x1_mm, x2_mm, y1_mm, y2_mm, rho_values))



def _ellipse_stationary_rhos(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
) -> list[float]:
    """Return real stationary points of ellipse area inside -1 < rho < 1."""
    qxx = 1.0 / (x1_mm * x2_mm)
    qyy = 1.0 / (y1_mm * y2_mm)
    linear_x = (x2_mm - x1_mm) * qxx
    linear_y = (y2_mm - y1_mm) * qyy
    u = linear_x / math.sqrt(qxx)
    v = linear_y / math.sqrt(qyy)
    sum_squares = u * u + v * v
    product = u * v

    # Derivative of A(rho) is zero at the real roots of this cubic.
    roots = np.roots([
        4.0,
        4.0 * product,
        -(4.0 + 3.0 * sum_squares),
        2.0 * product,
    ])
    output = [
        float(root.real)
        for root in roots
        if abs(root.imag) < 1e-9 and -1.0 < root.real < 1.0
    ]
    return sorted(output)


def minimum_area_ellipse_rho(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
) -> float:
    """Return the exact-family rho giving the global minimum ellipse area."""
    candidates = _ellipse_stationary_rhos(x1_mm, x2_mm, y1_mm, y2_mm)
    if not candidates:
        candidates = [0.0]
    areas = ellipse_family_arrays(
        x1_mm, x2_mm, y1_mm, y2_mm, np.asarray(candidates, dtype=float)
    )['area_mm2']
    return float(candidates[int(np.nanargmin(areas))])


def _axis_ratio_rho_limit(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    max_axis_ratio: float,
) -> Optional[float]:
    """Return the exact |rho| limit implied by an ellipse axis-ratio cap."""
    ratio = float(max_axis_ratio)
    qxx = 1.0 / (x1_mm * x2_mm)
    qyy = 1.0 / (y1_mm * y2_mm)
    trace = qxx + qyy
    contrast = qxx - qyy
    ratio_squared = ratio ** 2
    discriminant_fraction = (ratio_squared - 1.0) / (ratio_squared + 1.0)
    numerator = (discriminant_fraction * trace) ** 2 - contrast ** 2
    denominator = 4.0 * qxx * qyy
    rho_squared = numerator / denominator
    if rho_squared < -1e-12:
        return None
    return float(min(1.0 - 1e-12, math.sqrt(max(0.0, rho_squared))))


def ellipse_area_upper_bound_mm2(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    constraints: Mapping[str, Optional[float]],
) -> float:
    """
    Return a conservative finite area bound from explicit size/shape priors.

    A centre-offset constraint is applied when selecting feasible scenarios,
    but is deliberately not used by itself to claim a finite upper bound.
    """
    bounds: list[float] = []

    max_axis_ratio = constraints.get('max_axis_ratio')
    if max_axis_ratio is not None:
        rho_limit = _axis_ratio_rho_limit(
            x1_mm, x2_mm, y1_mm, y2_mm, float(max_axis_ratio)
        )
        if rho_limit is None:
            return np.nan
        candidates = [-rho_limit, rho_limit]
        candidates.extend(
            rho for rho in _ellipse_stationary_rhos(x1_mm, x2_mm, y1_mm, y2_mm)
            if -rho_limit <= rho <= rho_limit
        )
        areas = ellipse_family_arrays(
            x1_mm, x2_mm, y1_mm, y2_mm, np.asarray(candidates, dtype=float)
        )['area_mm2']
        bounds.append(float(np.nanmax(areas)))

    max_major_diameter_mm = constraints.get('max_major_diameter_mm')
    if max_major_diameter_mm is not None:
        # Any ellipse whose major diameter is at most D has area no greater
        # than a circle of diameter D. This is conservative but rigorous.
        bounds.append(math.pi * (float(max_major_diameter_mm) / 2.0) ** 2)

    if not bounds:
        return np.nan

    # The minimum of independently valid upper bounds remains a valid bound.
    # A tiny safety factor avoids accidental under-reporting from floating
    # point round-off in the analytic axis-ratio calculation.
    return float(min(bounds) * (1.0 + 1e-10))


def validate_caliper_arm_geometry(
    arm_azimuths_deg: Mapping[str, float] = CALIPER_ARM_AZIMUTHS_DEG,
) -> None:
    expected = {'X1': 0.0, 'X2': 180.0, 'Y1': 90.0, 'Y2': 270.0}
    if set(arm_azimuths_deg) != set(expected):
        raise ValueError(f'Arm azimuths must define exactly {sorted(expected)}.')
    for arm, expected_angle in expected.items():
        difference = ((float(arm_azimuths_deg[arm]) - expected_angle + 180.0) % 360.0) - 180.0
        if abs(difference) > 1e-9:
            raise NotImplementedError(
                'The closed-form ellipse family currently requires opposite, '
                'orthogonal arm pairs at 0°, 90°, 180° and 270°.'
            )


def ellipse_parameters_from_ranges(
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    rho: float,
) -> dict[str, float]:
    """Return centre, axes and orientation for one member of the feasible family."""
    family = ellipse_family_arrays(x1_mm, x2_mm, y1_mm, y2_mm, np.array([rho], dtype=float))
    qxx = 1.0 / (x1_mm * x2_mm)
    qyy = 1.0 / (y1_mm * y2_mm)
    qxy = rho * math.sqrt(qxx * qyy)
    quadratic = np.array([[qxx, qxy], [qxy, qyy]], dtype=float)
    eigenvalues, eigenvectors = np.linalg.eigh(quadratic)
    major_vector = eigenvectors[:, int(np.argmin(eigenvalues))]
    orientation_deg = (math.degrees(math.atan2(major_vector[1], major_vector[0])) + 180.0) % 180.0

    return {
        name: float(values[0]) for name, values in family.items()
    } | {'orientation_deg': orientation_deg}


def ellipse_boundary_xy(parameters: Mapping[str, float], samples: int = 241) -> tuple[np.ndarray, np.ndarray]:
    theta = np.linspace(0.0, 2.0 * math.pi, samples)
    angle = math.radians(float(parameters['orientation_deg']))
    cos_angle, sin_angle = math.cos(angle), math.sin(angle)
    a = float(parameters['semi_major_mm'])
    b = float(parameters['semi_minor_mm'])
    local_x = a * np.cos(theta)
    local_y = b * np.sin(theta)
    x = float(parameters['centre_x_mm']) + local_x * cos_angle - local_y * sin_angle
    y = float(parameters['centre_y_mm']) + local_x * sin_angle + local_y * cos_angle
    return x, y


def ellipse_contact_residuals_mm(
    parameters: Mapping[str, float],
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
) -> np.ndarray:
    """Return algebraic radial-equivalent residuals; values should be near zero."""
    angle = math.radians(float(parameters['orientation_deg']))
    cos_angle, sin_angle = math.cos(angle), math.sin(angle)
    centre = np.array([parameters['centre_x_mm'], parameters['centre_y_mm']], dtype=float)
    points = np.array([[x1_mm, 0.0], [-x2_mm, 0.0], [0.0, y1_mm], [0.0, -y2_mm]], dtype=float)
    shifted = points - centre
    local_x = shifted[:, 0] * cos_angle + shifted[:, 1] * sin_angle
    local_y = -shifted[:, 0] * sin_angle + shifted[:, 1] * cos_angle
    normalised_radius = np.sqrt(
        (local_x / float(parameters['semi_major_mm'])) ** 2
        + (local_y / float(parameters['semi_minor_mm'])) ** 2
    )
    return (normalised_radius - 1.0) * float(parameters['area_equivalent_diameter_mm']) / 2.0


def _ellipse_constraint_mask(
    family: Mapping[str, np.ndarray],
    constraints: Mapping[str, Optional[float]],
) -> np.ndarray:
    mask = np.isfinite(family['area_mm2'])
    if constraints['max_axis_ratio'] is not None:
        mask &= family['axis_ratio'] <= constraints['max_axis_ratio']
    if constraints['max_major_diameter_mm'] is not None:
        mask &= family['major_diameter_mm'] <= constraints['max_major_diameter_mm']
    if constraints['max_tool_to_ellipse_centre_mm'] is not None:
        mask &= family['tool_to_ellipse_centre_mm'] <= constraints['max_tool_to_ellipse_centre_mm']
    return mask


def _selected_ellipse_fields(
    prefix: str,
    x1_mm: float,
    x2_mm: float,
    y1_mm: float,
    y2_mm: float,
    rho: Optional[float],
) -> dict[str, float]:
    field_names = [
        'rho', 'area_mm2', 'area_equivalent_diameter_mm',
        'centre_x_mm', 'centre_y_mm', 'tool_to_ellipse_centre_mm',
        'semi_major_mm', 'semi_minor_mm', 'major_diameter_mm',
        'minor_diameter_mm', 'axis_ratio', 'orientation_deg',
    ]
    if rho is None or not np.isfinite(rho):
        return {f'{prefix}_{name}': np.nan for name in field_names}
    parameters = ellipse_parameters_from_ranges(x1_mm, x2_mm, y1_mm, y2_mm, float(rho))
    return {f'{prefix}_{name}': parameters[name] for name in field_names}


def prepare_caliper_uncertainty(
    log: LogRecord,
    frame: Optional[pd.DataFrame] = None,
    motor_current_min_ma: Optional[float] = 1.8,
    max_range_mm: float = 1_000.0,
    nominal_diameter_mm: Optional[float] = None,
    ellipse_constraints: Optional[Mapping[str, Optional[float]]] = None,
    rho_samples: int = 401,
    rho_margin: float = 1e-4,
    max_gap_m: Optional[float] = None,
) -> pd.DataFrame:
    """
    Prepare direct caliper observations and uncertainty-aware ellipse products.

    The returned lower/reference/upper products are model-qualified. They are
    not unique measured borehole geometry.
    """
    frame = log.df if frame is None else frame
    depth_col = physical_depth_column(log)
    columns = {'depth_m': depth_col}
    for arm in ARM_NAMES:
        curve = log.curve(arm)
        if curve is None:
            raise ValueError(f'{log.log_id} is missing {arm}.')
        columns[arm] = curve.column

    data = pd.DataFrame({
        name: pd.to_numeric(frame[column], errors='coerce')
        for name, column in columns.items()
    })
    motor_curve = log.curve('MotorCurrent')
    if motor_curve and motor_curve.column in frame:
        data['motor_current_ma'] = pd.to_numeric(frame[motor_curve.column], errors='coerce')

    valid = data[list(ARM_NAMES) + ['depth_m']].notna().all(axis=1)
    for arm in ARM_NAMES:
        valid &= data[arm].between(0.0, max_range_mm, inclusive='neither')
    if motor_current_min_ma is not None and 'motor_current_ma' in data:
        valid &= data['motor_current_ma'] >= motor_current_min_ma
    data = data.loc[valid].copy()
    if data.empty:
        raise ValueError('No valid caliper samples remain after QC filtering.')

    numeric_columns = [column for column in data.columns if column != 'depth_m']
    data = (
        data.groupby('depth_m', as_index=False)[numeric_columns]
        .median()
        .sort_values('depth_m')
        .reset_index(drop=True)
    )

    data['x_chord_mm'] = data['X1'] + data['X2']
    data['y_chord_mm'] = data['Y1'] + data['Y2']
    data['maximum_observed_chord_mm'] = data[['x_chord_mm', 'y_chord_mm']].max(axis=1)
    data['contact_polygon_area_mm2'] = 0.5 * data['x_chord_mm'] * data['y_chord_mm']
    data['contact_polygon_area_equivalent_diameter_mm'] = 2.0 * np.sqrt(
        data['contact_polygon_area_mm2'] / math.pi
    )

    constraints = _normalise_ellipse_constraints(ellipse_constraints)
    constraints_active = any(value is not None for value in constraints.values())
    # Axis-ratio and major-diameter limits mathematically bound the singular
    # family. A centre-offset limit alone is not sufficient in all geometries.
    size_or_shape_bound = (
        constraints['max_axis_ratio'] is not None
        or constraints['max_major_diameter_mm'] is not None
    )
    validate_caliper_arm_geometry()
    rho_grid = make_ellipse_rho_grid(rho_samples, rho_margin)

    ellipse_rows: list[dict[str, Any]] = []
    for row in data[list(ARM_NAMES)].itertuples(index=False, name=None):
        x1_mm, x2_mm, y1_mm, y2_mm = map(float, row)
        family = ellipse_family_arrays(x1_mm, x2_mm, y1_mm, y2_mm, rho_grid)
        absolute_min_rho = minimum_area_ellipse_rho(x1_mm, x2_mm, y1_mm, y2_mm)
        lower_rho = absolute_min_rho
        reference_rho = 0.0

        constraint_mask = _ellipse_constraint_mask(family, constraints)
        feasible_count = int(constraint_mask.sum())
        upper_scenario_rho: Optional[float] = None
        upper_bound_area_mm2 = np.nan

        if constraints_active and feasible_count == 0:
            upper_bound_finite = False
            constraint_status = 'NO_FEASIBLE_ELLIPSE'
        elif constraints_active and size_or_shape_bound:
            upper_bound_area_mm2 = ellipse_area_upper_bound_mm2(
                x1_mm, x2_mm, y1_mm, y2_mm, constraints
            )
            upper_bound_finite = bool(np.isfinite(upper_bound_area_mm2))
            if feasible_count:
                candidate_indices = np.flatnonzero(constraint_mask)
                scenario_index = int(
                    candidate_indices[np.argmax(family['area_mm2'][candidate_indices])]
                )
                upper_scenario_rho = float(family['rho'][scenario_index])
            constraint_status = (
                'FINITE_BOUND_FROM_EXPLICIT_PRIOR'
                if upper_bound_finite
                else 'NO_FEASIBLE_ELLIPSE'
            )
        else:
            upper_bound_finite = False
            constraint_status = (
                'UNBOUNDED_WITHOUT_SIZE_OR_SHAPE_PRIOR'
                if constraints_active
                else 'UNBOUNDED_UNCONSTRAINED_FAMILY'
            )

        output_row: dict[str, Any] = {
            'ellipse_family_unbounded_without_constraints': True,
            'ellipse_constraint_status': constraint_status,
            'ellipse_constraint_feasible_fraction': feasible_count / len(rho_grid) if constraints_active else 1.0,
            'ellipse_upper_bound_finite': upper_bound_finite,
            'upper_bound_area_mm2': upper_bound_area_mm2,
            'upper_bound_area_equivalent_diameter_mm': (
                2.0 * math.sqrt(upper_bound_area_mm2 / math.pi)
                if np.isfinite(upper_bound_area_mm2)
                else np.nan
            ),
        }
        output_row.update(_selected_ellipse_fields(
            'absolute_min', x1_mm, x2_mm, y1_mm, y2_mm, absolute_min_rho
        ))
        output_row.update(_selected_ellipse_fields(
            'lower', x1_mm, x2_mm, y1_mm, y2_mm, lower_rho
        ))
        output_row.update(_selected_ellipse_fields(
            'reference', x1_mm, x2_mm, y1_mm, y2_mm, reference_rho
        ))
        output_row.update(_selected_ellipse_fields(
            'upper_scenario', x1_mm, x2_mm, y1_mm, y2_mm, upper_scenario_rho
        ))
        ellipse_rows.append(output_row)

    data = pd.concat([data, pd.DataFrame(ellipse_rows, index=data.index)], axis=1)

    for prefix in ('contact_polygon', 'absolute_min', 'lower', 'reference', 'upper_scenario'):
        area_column = f'{prefix}_area_mm2' if prefix != 'contact_polygon' else 'contact_polygon_area_mm2'
        if area_column in data:
            data[f'{prefix}_area_m2'] = data[area_column] / 1_000_000.0
    data['upper_bound_area_m2'] = data['upper_bound_area_mm2'] / 1_000_000.0

    data['delta_depth_m'] = data['depth_m'].diff()
    positive_steps = data.loc[data['delta_depth_m'] > 0, 'delta_depth_m']
    median_step = float(positive_steps.median()) if not positive_steps.empty else np.nan
    if max_gap_m is None:
        max_gap_m = max(0.10, 5.0 * median_step) if np.isfinite(median_step) else 0.10
    data['segment_integrable'] = data['delta_depth_m'].between(0.0, max_gap_m, inclusive='right')

    area_products = {
        'contact_polygon_lower': 'contact_polygon_area_m2',
        'ellipse_lower': 'lower_area_m2',
        'ellipse_reference': 'reference_area_m2',
        'ellipse_upper_bound': 'upper_bound_area_m2',
    }
    for label, area_column in area_products.items():
        pair_valid = (
            data['segment_integrable']
            & data[area_column].notna()
            & data[area_column].shift(1).notna()
        )
        segment_area = (data[area_column] + data[area_column].shift(1)) / 2.0
        segment_column = f'segment_volume_{label}_m3'
        cumulative_column = f'cumulative_volume_{label}_m3'
        if label == 'ellipse_upper_bound':
            data[segment_column] = (segment_area * data['delta_depth_m']).where(pair_valid, np.nan)
            data[cumulative_column] = data[segment_column].fillna(0.0).cumsum()
        else:
            data[segment_column] = (segment_area * data['delta_depth_m']).where(pair_valid, 0.0)
            data[cumulative_column] = data[segment_column].cumsum()

    if nominal_diameter_mm is not None:
        nominal_area_m2 = math.pi * (nominal_diameter_mm / 2_000.0) ** 2
        data['nominal_area_m2'] = nominal_area_m2
        data['segment_volume_nominal_m3'] = (
            nominal_area_m2 * data['delta_depth_m']
        ).where(data['segment_integrable'], 0.0)
        data['cumulative_volume_nominal_m3'] = data['segment_volume_nominal_m3'].cumsum()

    data.attrs.update({
        'log_id': log.log_id,
        'calculation_profile': CALIPER_CALCULATION_PROFILE,
        'identifiability': 'one_parameter_ellipse_family',
        'median_step_m': median_step,
        'max_gap_m': max_gap_m,
        'nominal_diameter_mm': nominal_diameter_mm,
        'ellipse_constraints': constraints,
        'ellipse_constraints_label': ellipse_constraints_label(constraints),
        'rho_samples_requested': rho_samples,
        'rho_candidates_evaluated': len(rho_grid),
        'rho_margin': rho_margin,
    })
    return data


def caliper_uncertainty_summary(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
) -> pd.DataFrame:
    segment_length = geometry['delta_depth_m'].where(geometry['segment_integrable'], 0.0).fillna(0.0)
    total_length = float(segment_length.sum())
    lower_limit = nominal_diameter_mm - tolerance_mm
    upper_limit = nominal_diameter_mm + tolerance_mm

    lower_diameter = (
        geometry['lower_area_equivalent_diameter_mm']
        + geometry['lower_area_equivalent_diameter_mm'].shift(1)
    ) / 2.0
    upper_diameter = (
        geometry['upper_bound_area_equivalent_diameter_mm']
        + geometry['upper_bound_area_equivalent_diameter_mm'].shift(1)
    ) / 2.0
    reference_diameter = (
        geometry['reference_area_equivalent_diameter_mm']
        + geometry['reference_area_equivalent_diameter_mm'].shift(1)
    ) / 2.0
    chord = (
        geometry['maximum_observed_chord_mm']
        + geometry['maximum_observed_chord_mm'].shift(1)
    ) / 2.0

    finite_upper_segment = (
        geometry['ellipse_upper_bound_finite']
        & geometry['ellipse_upper_bound_finite'].shift(1, fill_value=False)
        & upper_diameter.notna()
    )
    definitely_above = lower_diameter > upper_limit
    definitely_below = finite_upper_segment & (upper_diameter < lower_limit)
    definitely_within = (
        finite_upper_segment
        & (lower_diameter >= lower_limit)
        & (upper_diameter <= upper_limit)
    )
    direct_chord_exceeds = chord > upper_limit
    indeterminate = geometry['segment_integrable'] & ~(
        definitely_above | definitely_below | definitely_within
    )

    ellipse_lower_volume = float(geometry['segment_volume_ellipse_lower_m3'].sum())
    reference_volume = float(geometry['segment_volume_ellipse_reference_m3'].sum())
    contact_polygon_volume = float(geometry['segment_volume_contact_polygon_lower_m3'].sum())
    nominal_volume = float(geometry.get('segment_volume_nominal_m3', pd.Series(0.0, index=geometry.index)).sum())

    expected_upper_segments = geometry['segment_integrable']
    upper_segments = geometry['segment_volume_ellipse_upper_bound_m3']
    all_upper_segments_finite = bool(
        expected_upper_segments.any()
        and upper_segments.loc[expected_upper_segments].notna().all()
    )
    upper_volume = float(upper_segments.loc[expected_upper_segments].sum()) if all_upper_segments_finite else np.nan

    definite_oversize_length = float(segment_length.where(definitely_above | direct_chord_exceeds, 0.0).sum())
    definite_undersize_length = float(segment_length.where(definitely_below, 0.0).sum())
    definite_within_length = float(segment_length.where(definitely_within, 0.0).sum())
    indeterminate_length = float(segment_length.where(indeterminate, 0.0).sum())

    if definite_oversize_length > 0:
        status = 'REVIEW — DEFINITE LOCAL OVERSIZE'
    elif definite_undersize_length > 0:
        status = 'REVIEW — DEFINITE LOCAL UNDERSIZE'
    elif all_upper_segments_finite and total_length and definite_within_length / total_length >= 0.95:
        status = 'PASS WITH EXPLICIT ELLIPSE PRIOR'
    else:
        status = 'INDETERMINATE — ADD CONSTRAINTS OR DATA'

    return pd.DataFrame([{
        'log_id': geometry.attrs.get('log_id'),
        'calculation_profile': geometry.attrs.get('calculation_profile'),
        'identifiability': 'UNDERDETERMINED',
        'ellipse_constraints': geometry.attrs.get('ellipse_constraints_label'),
        'valid_depth_from_m': float(geometry['depth_m'].min()),
        'valid_depth_to_m': float(geometry['depth_m'].max()),
        'integrated_length_m': total_length,
        'x_chord_median_mm': float(geometry['x_chord_mm'].median()),
        'y_chord_median_mm': float(geometry['y_chord_mm'].median()),
        'maximum_observed_chord_mm': float(geometry['maximum_observed_chord_mm'].max()),
        'ellipse_equivalent_diameter_lower_p10_mm': float(geometry['lower_area_equivalent_diameter_mm'].quantile(0.10)),
        'ellipse_equivalent_diameter_lower_median_mm': float(geometry['lower_area_equivalent_diameter_mm'].median()),
        'ellipse_equivalent_diameter_lower_p90_mm': float(geometry['lower_area_equivalent_diameter_mm'].quantile(0.90)),
        'ellipse_equivalent_diameter_reference_median_mm': float(geometry['reference_area_equivalent_diameter_mm'].median()),
        'ellipse_upper_bound_finite_for_full_interval': all_upper_segments_finite,
        'nominal_diameter_mm': nominal_diameter_mm,
        'diameter_tolerance_mm': tolerance_mm,
        'length_definitely_above_tolerance_m': float(segment_length.where(definitely_above, 0.0).sum()),
        'length_direct_chord_above_upper_tolerance_m': float(segment_length.where(direct_chord_exceeds, 0.0).sum()),
        'length_definitely_below_tolerance_m': definite_undersize_length,
        'length_definitely_within_tolerance_m': definite_within_length,
        'length_indeterminate_m': indeterminate_length,
        'contact_polygon_volume_lower_m3': contact_polygon_volume,
        'ellipse_model_volume_lower_m3': ellipse_lower_volume,
        'axis_aligned_reference_volume_m3': reference_volume,
        'ellipse_model_volume_upper_m3': upper_volume,
        'nominal_volume_m3': nominal_volume,
        'ellipse_volume_variance_lower_m3': ellipse_lower_volume - nominal_volume,
        'ellipse_volume_variance_reference_m3': reference_volume - nominal_volume,
        'ellipse_volume_variance_upper_m3': upper_volume - nominal_volume if np.isfinite(upper_volume) else np.nan,
        'reconciliation_status': status,
    }])


caliper_geometry = prepare_caliper_uncertainty(
    calibrated_log,
    motor_current_min_ma=CALIPER_MOTOR_CURRENT_MIN_MA,
    max_range_mm=CALIPER_MAX_RANGE_MM,
    nominal_diameter_mm=NOMINAL_DIAMETER_MM,
    ellipse_constraints=CALIPER_ELLIPSE_CONSTRAINTS,
    rho_samples=CALIPER_ELLIPSE_RHO_SAMPLES,
    rho_margin=CALIPER_ELLIPSE_RHO_MARGIN,
)
caliper_summary = caliper_uncertainty_summary(
    caliper_geometry,
    NOMINAL_DIAMETER_MM,
    DIAMETER_TOLERANCE_MM,
)

display(HTML(
    '<div class="ms-critical"><b>Geometry is not uniquely identifiable.</b> '
    'Opposite-arm sums are displayed as chords, and volume is reported as '
    'model-qualified bounds/scenarios. The unconstrained upper bound is infinite.</div>'
))
display(caliper_summary)

In [ ]:
def _format_optional(value: Any, pattern: str, missing: str = 'Unbounded') -> str:
    return format(float(value), pattern) if value is not None and np.isfinite(value) else missing


def metric_cards(summary: pd.Series) -> HTML:
    values = [
        ('Integrated length', f"{summary['integrated_length_m']:.2f} m"),
        ('Contact-polygon lower volume', f"{summary['contact_polygon_volume_lower_m3']:.3f} m³"),
        ('Ellipse-model lower volume', f"{summary['ellipse_model_volume_lower_m3']:.3f} m³"),
        ('Axis-aligned reference', f"{summary['axis_aligned_reference_volume_m3']:.3f} m³"),
        ('Ellipse upper volume', _format_optional(summary['ellipse_model_volume_upper_m3'], '.3f', 'Unbounded')),
        ('Status', str(summary['reconciliation_status'])),
    ]
    cards = ''.join(
        f'<div class="ms-card"><div class="label">{label}</div><div class="value">{value}</div></div>'
        for label, value in values
    )
    return HTML(cards)


display(metric_cards(caliper_summary.iloc[0]))


def caliper_interval_summary(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
    interval_m: float = 1.0,
) -> pd.DataFrame:
    output = geometry.copy()
    origin = math.floor(output['depth_m'].min() / interval_m) * interval_m
    output['interval_from_m'] = origin + np.floor((output['depth_m'] - origin) / interval_m) * interval_m
    output['interval_to_m'] = output['interval_from_m'] + interval_m

    grouped = output.groupby(['interval_from_m', 'interval_to_m'], as_index=False).agg(
        samples=('depth_m', 'size'),
        x_chord_mean_mm=('x_chord_mm', 'mean'),
        y_chord_mean_mm=('y_chord_mm', 'mean'),
        maximum_observed_chord_mm=('maximum_observed_chord_mm', 'max'),
        ellipse_equivalent_diameter_lower_mean_mm=('lower_area_equivalent_diameter_mm', 'mean'),
        ellipse_equivalent_diameter_reference_mean_mm=('reference_area_equivalent_diameter_mm', 'mean'),
        contact_polygon_volume_lower_m3=('segment_volume_contact_polygon_lower_m3', 'sum'),
        ellipse_model_volume_lower_m3=('segment_volume_ellipse_lower_m3', 'sum'),
        axis_aligned_reference_volume_m3=('segment_volume_ellipse_reference_m3', 'sum'),
        nominal_volume_m3=('segment_volume_nominal_m3', 'sum'),
    )

    upper_by_interval = (
        output.groupby(['interval_from_m', 'interval_to_m'])['segment_volume_ellipse_upper_bound_m3']
        .apply(lambda values: values.sum(min_count=1))
        .reset_index(name='ellipse_model_volume_upper_m3')
    )
    upper_diameter_by_interval = (
        output.groupby(['interval_from_m', 'interval_to_m'])['upper_bound_area_equivalent_diameter_mm']
        .apply(lambda values: values.mean() if values.notna().all() else np.nan)
        .reset_index(name='ellipse_equivalent_diameter_upper_mean_mm')
    )
    grouped = grouped.merge(upper_by_interval, on=['interval_from_m', 'interval_to_m'], how='left')
    grouped = grouped.merge(upper_diameter_by_interval, on=['interval_from_m', 'interval_to_m'], how='left')

    lower_limit = nominal_diameter_mm - tolerance_mm
    upper_limit = nominal_diameter_mm + tolerance_mm
    definite_oversize = (
        (grouped['ellipse_equivalent_diameter_lower_mean_mm'] > upper_limit)
        | (grouped['maximum_observed_chord_mm'] > upper_limit)
    )
    definite_undersize = (
        grouped['ellipse_equivalent_diameter_upper_mean_mm'].notna()
        & (grouped['ellipse_equivalent_diameter_upper_mean_mm'] < lower_limit)
    )
    within_with_prior = (
        grouped['ellipse_equivalent_diameter_upper_mean_mm'].notna()
        & (grouped['ellipse_equivalent_diameter_lower_mean_mm'] >= lower_limit)
        & (grouped['ellipse_equivalent_diameter_upper_mean_mm'] <= upper_limit)
    )
    grouped['status'] = np.select(
        [definite_oversize, definite_undersize, within_with_prior],
        ['REVIEW — DEFINITE OVERSIZE', 'REVIEW — DEFINITE UNDERSIZE', 'WITHIN TOLERANCE WITH PRIOR'],
        default='INDETERMINATE',
    )
    return grouped


interval_summary = caliper_interval_summary(
    caliper_geometry,
    NOMINAL_DIAMETER_MM,
    DIAMETER_TOLERANCE_MM,
)
display(interval_summary)

## 10. Caliper visualisations

The plots in this section deliberately separate direct measurements from model-dependent products:

- arm ranges and tool-axis chords are direct observations;
- the contact polygon is constructed directly from the four contacts;
- minimum/reference/upper ellipses are members of an assumed ellipse family;
- the reference tube is a scenario visualisation and must not be interpreted as a reconstructed borehole wall.

In [ ]:
def plot_caliper_uncertainty_dashboard(
    geometry: pd.DataFrame,
    nominal_diameter_mm: float,
    tolerance_mm: float,
) -> go.Figure:
    fig = make_subplots(
        rows=1,
        cols=5,
        shared_yaxes=True,
        horizontal_spacing=0.035,
        subplot_titles=(
            'Tool-to-wall ranges',
            'Observed chords (not diameters)',
            'Area-equivalent diameter',
            'Cross-sectional area products',
            'Cumulative volume products',
        ),
    )
    depth = geometry['depth_m']

    for index, arm in enumerate(ARM_NAMES):
        fig.add_trace(go.Scatter(
            x=geometry[arm], y=depth, mode='lines', name=arm,
            line={'width': 1.2, 'color': MS_COLOURS[index]},
        ), row=1, col=1)

    for index, column in enumerate(['x_chord_mm', 'y_chord_mm']):
        fig.add_trace(go.Scatter(
            x=geometry[column], y=depth, mode='lines', name=column,
            line={'width': 1.5, 'color': MS_COLOURS[index + 1]},
        ), row=1, col=2)

    for value, dash, name in [
        (nominal_diameter_mm, 'solid', 'Nominal diameter scale'),
        (nominal_diameter_mm - tolerance_mm, 'dot', 'Lower tolerance scale'),
        (nominal_diameter_mm + tolerance_mm, 'dot', 'Upper tolerance scale'),
    ]:
        fig.add_trace(go.Scatter(
            x=np.full(len(depth), value), y=depth, mode='lines', name=name,
            line={'width': 1.0, 'dash': dash, 'color': '#59636E'},
        ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=geometry['lower_area_equivalent_diameter_mm'], y=depth,
        mode='lines', name='Ellipse lower bound',
        line={'width': 2.0, 'color': MS_COLOURS[1]},
    ), row=1, col=3)
    fig.add_trace(go.Scatter(
        x=geometry['reference_area_equivalent_diameter_mm'], y=depth,
        mode='lines', name='Axis-aligned reference',
        line={'width': 1.6, 'dash': 'dash', 'color': MS_COLOURS[2]},
    ), row=1, col=3)

    if geometry['upper_bound_area_equivalent_diameter_mm'].notna().any():
        fig.add_trace(go.Scatter(
            x=geometry['upper_bound_area_equivalent_diameter_mm'], y=depth,
            mode='lines', name='Conservative constrained upper bound',
            line={'width': 1.4, 'color': MS_COLOURS[5]},
        ), row=1, col=3)

    for value, dash, name in [
        (nominal_diameter_mm, 'solid', 'Nominal'),
        (nominal_diameter_mm - tolerance_mm, 'dot', 'Lower tolerance'),
        (nominal_diameter_mm + tolerance_mm, 'dot', 'Upper tolerance'),
    ]:
        fig.add_trace(go.Scatter(
            x=np.full(len(depth), value), y=depth, mode='lines', name=name,
            line={'width': 1.0, 'dash': dash, 'color': '#59636E'},
        ), row=1, col=3)

    for column, name, colour, dash in [
        ('contact_polygon_area_m2', 'Contact polygon', MS_COLOURS[3], 'dot'),
        ('lower_area_m2', 'Ellipse lower', MS_COLOURS[1], 'solid'),
        ('reference_area_m2', 'Axis-aligned reference', MS_COLOURS[2], 'dash'),
    ]:
        fig.add_trace(go.Scatter(
            x=geometry[column], y=depth, mode='lines', name=name,
            line={'width': 1.5, 'color': colour, 'dash': dash},
        ), row=1, col=4)
    if geometry['upper_bound_area_m2'].notna().any():
        fig.add_trace(go.Scatter(
            x=geometry['upper_bound_area_m2'], y=depth, mode='lines', name='Conservative upper area bound',
            line={'width': 1.4, 'color': MS_COLOURS[5]},
        ), row=1, col=4)

    for column, name, colour, dash in [
        ('cumulative_volume_contact_polygon_lower_m3', 'Contact-polygon lower', MS_COLOURS[3], 'dot'),
        ('cumulative_volume_ellipse_lower_m3', 'Ellipse-model lower', MS_COLOURS[1], 'solid'),
        ('cumulative_volume_ellipse_reference_m3', 'Axis-aligned reference', MS_COLOURS[2], 'dash'),
        ('cumulative_volume_nominal_m3', 'Nominal', MS_COLOURS[4], 'dash'),
    ]:
        fig.add_trace(go.Scatter(
            x=geometry[column], y=depth, mode='lines', name=name,
            line={'width': 1.6, 'color': colour, 'dash': dash},
        ), row=1, col=5)
    if geometry['upper_bound_area_m2'].notna().any():
        fig.add_trace(go.Scatter(
            x=geometry['cumulative_volume_ellipse_upper_bound_m3'], y=depth,
            mode='lines', name='Conservative upper volume bound',
            line={'width': 1.5, 'color': MS_COLOURS[5]},
        ), row=1, col=5)

    fig.update_yaxes(title_text='Measured depth (m)', autorange='reversed', row=1, col=1)
    fig.update_xaxes(title_text='Range (mm)', row=1, col=1)
    fig.update_xaxes(title_text='Chord length (mm)', row=1, col=2)
    fig.update_xaxes(title_text='Area-equivalent diameter (mm)', row=1, col=3)
    fig.update_xaxes(title_text='Area (m²)', row=1, col=4)
    fig.update_xaxes(title_text='Volume (m³)', row=1, col=5)

    upper_label = (
        geometry.attrs.get('ellipse_constraints_label')
        if geometry['upper_bound_area_m2'].notna().any()
        else 'upper bound unbounded'
    )
    fig.update_layout(
        title=(
            f"{geometry.attrs.get('log_id')} — caliper observations and uncertainty "
            f"({upper_label})"
        ),
        template='plotly_white', height=840, width=1600, hovermode='y unified',
        legend={'orientation': 'h', 'y': -0.13},
        margin={'l': 70, 'r': 30, 't': 100, 'b': 125},
    )
    return fig


plot_caliper_uncertainty_dashboard(
    caliper_geometry,
    NOMINAL_DIAMETER_MM,
    DIAMETER_TOLERANCE_MM,
).show()

In [ ]:
def nearest_caliper_row(geometry: pd.DataFrame, target_depth_m: float) -> pd.Series:
    index = (geometry['depth_m'] - target_depth_m).abs().idxmin()
    return geometry.loc[index]


def _contact_points_from_row(row: pd.Series) -> np.ndarray:
    return np.array([
        [float(row['X1']), 0.0],
        [0.0, float(row['Y1'])],
        [-float(row['X2']), 0.0],
        [0.0, -float(row['Y2'])],
    ])


def plot_caliper_cross_section_family(
    geometry: pd.DataFrame,
    target_depth_m: float,
    nominal_diameter_mm: Optional[float] = None,
    display_rhos: Sequence[float] = CALIPER_FAMILY_DISPLAY_RHOS,
) -> go.Figure:
    row = nearest_caliper_row(geometry, target_depth_m)
    ranges = tuple(float(row[name]) for name in ARM_NAMES)
    contacts = _contact_points_from_row(row)
    closed_contacts = np.vstack([contacts, contacts[0]])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=closed_contacts[:, 0], y=closed_contacts[:, 1],
        mode='lines', fill='toself', name='Contact polygon',
        line={'width': 1.5, 'color': MS_COLOURS[3]},
        fillcolor='rgba(233,196,106,0.18)',
    ))

    for rho in display_rhos:
        if not -1.0 < float(rho) < 1.0:
            continue
        parameters = ellipse_parameters_from_ranges(*ranges, float(rho))
        x, y = ellipse_boundary_xy(parameters)
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines',
            name=f"Feasible ρ={rho:+.2f}; A={parameters['area_mm2'] / 1_000:.1f}×10³ mm²",
            line={'width': 1.0, 'color': '#9AA5B1'}, opacity=0.45,
            hovertemplate=(
                f"ρ={rho:+.3f}<br>Area={parameters['area_mm2']:.1f} mm²"
                f"<br>Deq={parameters['area_equivalent_diameter_mm']:.1f} mm"
                f"<br>Axis ratio={parameters['axis_ratio']:.2f}<extra></extra>"
            ),
        ))

    selected = [
        ('lower', 'Minimum/constrained lower ellipse', MS_COLOURS[1], 'solid', 3.0),
        ('reference', 'Axis-aligned reference ellipse', MS_COLOURS[2], 'dash', 2.5),
    ]
    if pd.notna(row['upper_scenario_rho']):
        selected.append(('upper_scenario', 'High-area feasible scenario', MS_COLOURS[5], 'solid', 2.5))

    for prefix, name, colour, dash, width in selected:
        rho = float(row[f'{prefix}_rho'])
        parameters = ellipse_parameters_from_ranges(*ranges, rho)
        x, y = ellipse_boundary_xy(parameters)
        fig.add_trace(go.Scatter(
            x=x, y=y, mode='lines', name=name,
            line={'width': width, 'color': colour, 'dash': dash},
            hovertemplate=(
                f"{name}<br>ρ={rho:+.4f}"
                f"<br>Area={parameters['area_mm2']:.1f} mm²"
                f"<br>Deq={parameters['area_equivalent_diameter_mm']:.1f} mm"
                f"<br>Major={parameters['major_diameter_mm']:.1f} mm"
                f"<br>Minor={parameters['minor_diameter_mm']:.1f} mm"
                f"<br>Axis ratio={parameters['axis_ratio']:.2f}<extra></extra>"
            ),
        ))
        fig.add_trace(go.Scatter(
            x=[parameters['centre_x_mm']], y=[parameters['centre_y_mm']],
            mode='markers', name=f'{name} centre', showlegend=False,
            marker={'size': 7, 'symbol': 'diamond', 'color': colour},
            hovertemplate=f"{name} centre<extra></extra>",
        ))

    if nominal_diameter_mm is not None:
        reference_parameters = ellipse_parameters_from_ranges(*ranges, float(row['reference_rho']))
        theta = np.linspace(0.0, 2.0 * math.pi, 241)
        radius = nominal_diameter_mm / 2.0
        fig.add_trace(go.Scatter(
            x=reference_parameters['centre_x_mm'] + radius * np.cos(theta),
            y=reference_parameters['centre_y_mm'] + radius * np.sin(theta),
            mode='lines', name='Nominal circle centred on reference scenario',
            line={'width': 1.3, 'dash': 'dot', 'color': MS_COLOURS[4]},
        ))

    arm_points = {
        'X1': (float(row['X1']), 0.0),
        'X2': (-float(row['X2']), 0.0),
        'Y1': (0.0, float(row['Y1'])),
        'Y2': (0.0, -float(row['Y2'])),
    }
    for arm, (x, y) in arm_points.items():
        fig.add_trace(go.Scatter(
            x=[0.0, x], y=[0.0, y], mode='lines+markers+text',
            text=['', arm], textposition='top center', name=arm,
            line={'width': 1.2}, marker={'size': [6, 9]},
        ))
    fig.add_trace(go.Scatter(
        x=[0.0], y=[0.0], mode='markers', name='Tool reference',
        marker={'size': 12, 'symbol': 'x', 'color': '#111111'},
    ))

    upper_text = (
        f"conservative upper-bound Deq={row['upper_bound_area_equivalent_diameter_mm']:.1f} mm"
        if pd.notna(row['upper_bound_area_equivalent_diameter_mm'])
        else 'upper area unbounded without a size/shape prior'
    )
    fig.update_xaxes(title='Tool X coordinate (mm)', scaleanchor='y', scaleratio=1, zeroline=True)
    fig.update_yaxes(title='Tool Y coordinate (mm)', zeroline=True)
    fig.update_layout(
        title=(
            f"Feasible ellipse family at {row['depth_m']:.3f} m — "
            f"lower Deq={row['lower_area_equivalent_diameter_mm']:.1f} mm; "
            f"reference Deq={row['reference_area_equivalent_diameter_mm']:.1f} mm; {upper_text}"
        ),
        template='plotly_white', width=940, height=800,
        legend={'orientation': 'h', 'y': -0.16},
        margin={'l': 70, 'r': 30, 't': 100, 'b': 145},
    )
    return fig


CROSS_SECTION_DEPTH_M = float(caliper_geometry['depth_m'].median())
plot_caliper_cross_section_family(
    caliper_geometry,
    CROSS_SECTION_DEPTH_M,
    NOMINAL_DIAMETER_MM,
).show()

In [ ]:
def plot_caliper_contact_traces_3d(
    geometry: pd.DataFrame,
    max_stations: int = 160,
    scenario: Optional[str] = 'reference',
    angular_samples: int = 42,
) -> go.Figure:
    """
    Plot direct tool-relative contact traces and, optionally, one translucent
    ellipse scenario. The scenario surface is not a uniquely resolved wall.
    """
    if scenario not in {None, 'lower', 'reference', 'upper_scenario'}:
        raise ValueError("scenario must be None, 'lower', 'reference' or 'upper_scenario'.")

    stride = max(1, math.ceil(len(geometry) / max_stations))
    sample = geometry.iloc[::stride].copy()
    fig = go.Figure()

    arm_coordinates = {
        'X1': (sample['X1'] / 1000.0, np.zeros(len(sample))),
        'X2': (-sample['X2'] / 1000.0, np.zeros(len(sample))),
        'Y1': (np.zeros(len(sample)), sample['Y1'] / 1000.0),
        'Y2': (np.zeros(len(sample)), -sample['Y2'] / 1000.0),
    }
    for index, (arm, (x, y)) in enumerate(arm_coordinates.items()):
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=sample['depth_m'], mode='lines', name=f'{arm} contacts',
            line={'width': 5, 'color': MS_COLOURS[index]},
        ))
    fig.add_trace(go.Scatter3d(
        x=np.zeros(len(sample)), y=np.zeros(len(sample)), z=sample['depth_m'],
        mode='lines', name='Tool reference line', line={'width': 6, 'color': '#111111'},
    ))

    scenario_label = 'direct contacts only'
    if scenario is not None:
        required = [
            f'{scenario}_centre_x_mm', f'{scenario}_centre_y_mm',
            f'{scenario}_semi_major_mm', f'{scenario}_semi_minor_mm',
            f'{scenario}_orientation_deg',
        ]
        scenario_sample = sample.dropna(subset=required)
        if not scenario_sample.empty:
            theta = np.linspace(0.0, 2.0 * math.pi, angular_samples)
            angles = np.deg2rad(scenario_sample[f'{scenario}_orientation_deg'].to_numpy())[:, None]
            a = scenario_sample[f'{scenario}_semi_major_mm'].to_numpy()[:, None] / 1000.0
            b = scenario_sample[f'{scenario}_semi_minor_mm'].to_numpy()[:, None] / 1000.0
            local_x = a * np.cos(theta)[None, :]
            local_y = b * np.sin(theta)[None, :]
            centre_x = scenario_sample[f'{scenario}_centre_x_mm'].to_numpy()[:, None] / 1000.0
            centre_y = scenario_sample[f'{scenario}_centre_y_mm'].to_numpy()[:, None] / 1000.0
            x = centre_x + local_x * np.cos(angles) - local_y * np.sin(angles)
            y = centre_y + local_x * np.sin(angles) + local_y * np.cos(angles)
            z = np.repeat(scenario_sample['depth_m'].to_numpy()[:, None], angular_samples, axis=1)
            fig.add_trace(go.Surface(
                x=x, y=y, z=z, surfacecolor=z, colorscale='Viridis',
                opacity=0.22, showscale=False, name=f'{scenario} ellipse scenario',
                hoverinfo='skip',
            ))
            scenario_label = f'{scenario} ellipse scenario — not uniquely resolved'

    fig.update_layout(
        title=f'3D tool-relative caliper contacts with {scenario_label}',
        template='plotly_white', width=980, height=780,
        scene={
            'xaxis_title': 'Tool X (m)',
            'yaxis_title': 'Tool Y (m)',
            'zaxis_title': 'Measured depth (m)',
            'zaxis': {'autorange': 'reversed'},
            'aspectmode': 'manual',
            'aspectratio': {'x': 1, 'y': 1, 'z': 4},
        },
        margin={'l': 20, 'r': 20, 't': 80, 'b': 20},
    )
    return fig


display(HTML(
    '<div class="ms-warning"><b>3D interpretation.</b> The four coloured traces are direct '
    'tool-relative contacts. The translucent surface is the axis-aligned reference scenario, '
    'not a measured or uniquely reconstructed borehole wall.</div>'
))
plot_caliper_contact_traces_3d(caliper_geometry, scenario='reference').show()

## 11. Hole plan / inspection register and uncertainty-aware reconciliation

A planned diameter can still be reconciled against the caliper observations, but the outcome must distinguish what is directly demonstrated from what depends on an ellipse prior:

- a chord longer than the upper diameter tolerance is direct evidence of a local width exceeding that tolerance;
- an ellipse-model lower bound above the upper tolerance is definite oversize under the ellipse assumption;
- undersize or a pass generally cannot be established without a finite upper bound;
- a pass based on a constrained ellipse family must retain the constraint profile and be labelled accordingly.

In [ ]:
# Replace this example with a CSV or database extract from the drill-and-blast plan.
hole_register = pd.DataFrame([
    {
        'hole_id': calibrated_id,
        'planned_depth_m': 28.0,
        'nominal_diameter_mm': NOMINAL_DIAMETER_MM,
        'diameter_tolerance_mm': DIAMETER_TOLERANCE_MM,
        'ellipse_max_axis_ratio': CALIPER_ELLIPSE_CONSTRAINTS['max_axis_ratio'],
        'ellipse_max_major_diameter_mm': CALIPER_ELLIPSE_CONSTRAINTS['max_major_diameter_mm'],
        'ellipse_max_tool_to_centre_mm': CALIPER_ELLIPSE_CONSTRAINTS['max_tool_to_ellipse_centre_mm'],
        'planned_inclination_from_vertical_deg': 0.0,
        'planned_azimuth_deg': 0.0,
        'collar_easting_m': np.nan,
        'collar_northing_m': np.nan,
        'collar_elevation_m': np.nan,
    }
])
display(hole_register)


def _optional_register_float(row: pd.Series, column: str) -> Optional[float]:
    value = row.get(column, np.nan)
    return float(value) if pd.notna(value) else None


def reconcile_caliper_to_register(
    log: LogRecord,
    register_row: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    constraints = {
        'max_axis_ratio': _optional_register_float(register_row, 'ellipse_max_axis_ratio'),
        'max_major_diameter_mm': _optional_register_float(register_row, 'ellipse_max_major_diameter_mm'),
        'max_tool_to_ellipse_centre_mm': _optional_register_float(register_row, 'ellipse_max_tool_to_centre_mm'),
    }
    geometry = prepare_caliper_uncertainty(
        log,
        motor_current_min_ma=CALIPER_MOTOR_CURRENT_MIN_MA,
        max_range_mm=CALIPER_MAX_RANGE_MM,
        nominal_diameter_mm=float(register_row['nominal_diameter_mm']),
        ellipse_constraints=constraints,
        rho_samples=CALIPER_ELLIPSE_RHO_SAMPLES,
        rho_margin=CALIPER_ELLIPSE_RHO_MARGIN,
    )
    summary = caliper_uncertainty_summary(
        geometry,
        nominal_diameter_mm=float(register_row['nominal_diameter_mm']),
        tolerance_mm=float(register_row['diameter_tolerance_mm']),
    )
    summary['planned_depth_m'] = float(register_row['planned_depth_m'])
    summary['logged_to_depth_m'] = float(geometry['depth_m'].max())
    summary['depth_variance_m'] = summary['logged_to_depth_m'] - summary['planned_depth_m']
    return geometry, summary


register_geometry, register_reconciliation = reconcile_caliper_to_register(
    calibrated_log,
    hole_register.loc[hole_register['hole_id'].eq(calibrated_id)].iloc[0],
)
display(register_reconciliation)

## 12. Directional survey and 3D borehole trajectory

In [ ]:
def compute_minimum_curvature_trajectory(
    log: LogRecord,
    inclination_reference: str = 'vertical',
) -> pd.DataFrame:
    depth_col = physical_depth_column(log)
    inclination_col = find_curve_column(log, 'Incline', 'Verticality') or find_curve_column(log, 'Incline')
    azimuth_col = find_curve_column(log, 'Azimuth', 'Verticality') or find_curve_column(log, 'Azimuth')
    if not inclination_col or not azimuth_col:
        raise ValueError(f'{log.log_id} does not contain inclination and azimuth curves.')

    survey = pd.DataFrame({
        'measured_depth_m': pd.to_numeric(log.df[depth_col], errors='coerce'),
        'inclination_deg': pd.to_numeric(log.df[inclination_col], errors='coerce'),
        'azimuth_deg': pd.to_numeric(log.df[azimuth_col], errors='coerce'),
    }).dropna()
    survey = survey.groupby('measured_depth_m', as_index=False).median().sort_values('measured_depth_m').reset_index(drop=True)
    survey = survey.loc[survey['measured_depth_m'].diff().fillna(1.0) > 0].reset_index(drop=True)
    if len(survey) < 2:
        raise ValueError('At least two valid directional stations are required.')

    inclination = survey['inclination_deg'].to_numpy(dtype=float)
    if inclination_reference.casefold() == 'horizontal':
        inclination = 90.0 - inclination
    elif inclination_reference.casefold() != 'vertical':
        raise ValueError("inclination_reference must be 'vertical' or 'horizontal'.")

    inc = np.deg2rad(inclination)
    azi = np.deg2rad(np.mod(survey['azimuth_deg'].to_numpy(dtype=float), 360.0))
    md = survey['measured_depth_m'].to_numpy(dtype=float)
    dmd = np.diff(md)

    inc1, inc2 = inc[:-1], inc[1:]
    azi1, azi2 = azi[:-1], azi[1:]
    cosine_dogleg = np.cos(inc1) * np.cos(inc2) + np.sin(inc1) * np.sin(inc2) * np.cos(azi2 - azi1)
    dogleg = np.arccos(np.clip(cosine_dogleg, -1.0, 1.0))
    ratio_factor = np.ones_like(dogleg)
    nonzero = dogleg > 1e-10
    ratio_factor[nonzero] = 2.0 * np.tan(dogleg[nonzero] / 2.0) / dogleg[nonzero]

    d_north = 0.5 * dmd * (np.sin(inc1) * np.cos(azi1) + np.sin(inc2) * np.cos(azi2)) * ratio_factor
    d_east = 0.5 * dmd * (np.sin(inc1) * np.sin(azi1) + np.sin(inc2) * np.sin(azi2)) * ratio_factor
    d_tvd = 0.5 * dmd * (np.cos(inc1) + np.cos(inc2)) * ratio_factor

    survey['inclination_from_vertical_deg'] = inclination
    survey['northing_offset_m'] = np.r_[0.0, np.cumsum(d_north)]
    survey['easting_offset_m'] = np.r_[0.0, np.cumsum(d_east)]
    survey['tvd_m'] = np.r_[0.0, np.cumsum(d_tvd)]
    survey['horizontal_departure_m'] = np.hypot(survey['easting_offset_m'], survey['northing_offset_m'])
    survey['dogleg_deg'] = np.r_[np.nan, np.rad2deg(dogleg)]
    survey['dogleg_severity_deg_per_30m'] = np.r_[np.nan, np.rad2deg(dogleg) / dmd * 30.0]
    survey.attrs['log_id'] = log.log_id
    return survey


def trajectory_summary(trajectory: pd.DataFrame) -> pd.DataFrame:
    toe = trajectory.iloc[-1]
    return pd.DataFrame([{
        'log_id': trajectory.attrs.get('log_id'),
        'survey_from_md_m': float(trajectory['measured_depth_m'].min()),
        'survey_to_md_m': float(trajectory['measured_depth_m'].max()),
        'survey_interval_m': float(trajectory['measured_depth_m'].max() - trajectory['measured_depth_m'].min()),
        'toe_tvd_m': float(toe['tvd_m']),
        'toe_easting_offset_m': float(toe['easting_offset_m']),
        'toe_northing_offset_m': float(toe['northing_offset_m']),
        'toe_horizontal_departure_m': float(toe['horizontal_departure_m']),
        'maximum_dogleg_severity_deg_per_30m': float(trajectory['dogleg_severity_deg_per_30m'].max()),
    }])


def plot_trajectory(trajectory: pd.DataFrame) -> go.Figure:
    fig = go.Figure(go.Scatter3d(
        x=trajectory['easting_offset_m'], y=trajectory['northing_offset_m'], z=trajectory['tvd_m'],
        mode='lines+markers', name='Measured trajectory',
        line={'width': 6, 'color': MS_COLOURS[1]}, marker={'size': 2},
        customdata=np.column_stack([
            trajectory['measured_depth_m'], trajectory['inclination_from_vertical_deg'], trajectory['azimuth_deg']
        ]),
        hovertemplate='E: %{x:.3f} m<br>N: %{y:.3f} m<br>TVD: %{z:.3f} m<br>MD: %{customdata[0]:.3f} m<br>Inc: %{customdata[1]:.2f}°<br>Azi: %{customdata[2]:.2f}°<extra></extra>',
    ))
    fig.update_layout(
        title=f"{trajectory.attrs.get('log_id')} — minimum-curvature trajectory",
        template='plotly_white', width=900, height=760,
        scene={
            'xaxis_title': 'Easting offset (m)', 'yaxis_title': 'Northing offset (m)', 'zaxis_title': 'TVD down (m)',
            'zaxis': {'autorange': 'reversed'}, 'aspectmode': 'data',
        }, margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
    )
    return fig

trajectory = None
if directional_example_id:
    trajectory = compute_minimum_curvature_trajectory(directional_example, TRAJECTORY_INCLINATION_REFERENCE)
    display(trajectory_summary(trajectory))
    plot_trajectory(trajectory).show()

The trajectory calculation assumes inclination is measured from vertical and azimuth is clockwise from north. Those are common survey conventions, but the app should store the convention explicitly with every data source and never infer it silently.

> **Near-vertical survey caution:** the selected ACS run has a median inclination of about 0.2°. At very low inclination, azimuth is poorly constrained and small station-to-station changes can create a very large normalised dogleg-severity value even though the computed toe departure is only about 0.04 m. For these holes, review inclination, total departure and instrument QC together; do not use the maximum DLS in isolation.

## 13. Repeated-run comparison for calibration and field repeatability

In [ ]:
def interpolate_track(log: LogRecord, column: str, grid: np.ndarray) -> np.ndarray:
    depth_col = physical_depth_column(log)
    data = pd.DataFrame({
        'depth': pd.to_numeric(log.df[depth_col], errors='coerce'),
        'value': pd.to_numeric(log.df[column], errors='coerce'),
    }).dropna()
    data = data.groupby('depth', as_index=False).median().sort_values('depth')
    return np.interp(grid, data['depth'], data['value'])


def repeated_run_matrix(
    selected_logs: Sequence[LogRecord],
    mnemonic: str,
    grid_step_m: float = 0.05,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prepared = []
    for log in selected_logs:
        column = find_curve_column(log, mnemonic)
        if not column:
            continue
        depth = pd.to_numeric(log.df[physical_depth_column(log)], errors='coerce').dropna()
        values = pd.to_numeric(log.df[column], errors='coerce').dropna()
        if len(depth) < 2 or len(values) < 2:
            continue
        prepared.append((log, column, float(depth.min()), float(depth.max())))
    if len(prepared) < 2:
        raise ValueError('At least two comparable runs are required.')

    lower = max(item[2] for item in prepared)
    upper = min(item[3] for item in prepared)
    grid = np.arange(lower, upper + grid_step_m / 2.0, grid_step_m)
    matrix = pd.DataFrame({'depth_m': grid})
    for log, column, _, _ in prepared:
        matrix[log.log_id] = interpolate_track(log, column, grid)
    value_columns = [column for column in matrix.columns if column != 'depth_m']
    matrix['ensemble_mean'] = matrix[value_columns].mean(axis=1)
    matrix['ensemble_std'] = matrix[value_columns].std(axis=1)

    metrics = []
    for column in value_columns:
        residual = matrix[column] - matrix['ensemble_mean']
        metrics.append({
            'log_id': column,
            'bias_to_ensemble': float(residual.mean()),
            'MAE_to_ensemble': float(residual.abs().mean()),
            'RMSE_to_ensemble': float(np.sqrt(np.mean(residual ** 2))),
            'correlation_to_ensemble': float(matrix[[column, 'ensemble_mean']].corr().iloc[0, 1]),
        })
    return matrix, pd.DataFrame(metrics)


def plot_repeated_runs(matrix: pd.DataFrame, mnemonic: str, unit: str = '') -> go.Figure:
    value_columns = [column for column in matrix.columns if column not in {'depth_m', 'ensemble_mean', 'ensemble_std'}]
    fig = go.Figure()
    for index, column in enumerate(value_columns):
        fig.add_trace(go.Scatter(x=matrix[column], y=matrix['depth_m'], mode='lines', name=column,
                                 opacity=0.48, line={'width': 1.0, 'color': MS_COLOURS[index % len(MS_COLOURS)]}))
    upper = matrix['ensemble_mean'] + 2.0 * matrix['ensemble_std']
    lower = matrix['ensemble_mean'] - 2.0 * matrix['ensemble_std']
    fig.add_trace(go.Scatter(x=upper, y=matrix['depth_m'], mode='lines', line={'width': 0}, showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=lower, y=matrix['depth_m'], mode='lines', fill='tonextx', name='Ensemble ±2σ',
                             line={'width': 0}, fillcolor='rgba(42,157,143,0.18)', hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=matrix['ensemble_mean'], y=matrix['depth_m'], mode='lines', name='Ensemble mean',
                             line={'width': 3, 'color': '#111111'}))
    fig.update_yaxes(title='Measured depth (m)', autorange='reversed')
    fig.update_xaxes(title=f'{mnemonic} {f"({unit})" if unit else ""}')
    fig.update_layout(title=f'Repeated-run comparison — {mnemonic}', template='plotly_white', width=900, height=760,
                      legend={'orientation': 'h', 'y': -0.14}, margin={'l': 70, 'r': 30, 't': 80, 'b': 125})
    return fig

# Use the first seven upward ACS runs: they overlap over a useful interval and contain gamma data.
repeat_logs = [
    log for key, log in logs.items()
    if re.fullmatch(r'ACS-03_Run[1-7]', Path(key).name) and (log.view_type or '').casefold() == 'up'
]
if len(repeat_logs) >= 2:
    gamma_matrix, gamma_repeatability = repeated_run_matrix(repeat_logs, 'GR', grid_step_m=0.05)
    display(gamma_repeatability)
    gamma_unit = curve_unit(repeat_logs[0], find_curve_column(repeat_logs[0], 'GR'))
    plot_repeated_runs(gamma_matrix, 'GR', gamma_unit).show()

## 14. Export-ready data products for the MiningSim application

The application hand-off now preserves the distinction between observation and inference. Raw ranges and chords are exported separately from ellipse-family products. Every model-dependent row carries the calculation profile, constraint profile and upper-bound status so a downstream user cannot mistake a reference scenario for measured geometry.

Legacy fields such as `diameter_x_mm`, `diameter_y_mm`, `ovality_pct`, `hole_centre_x_mm`, `tool_offset_mm` and `measured_volume_m3` are intentionally omitted because they implied a unique geometry that the four contacts do not resolve.

In [ ]:
def build_log_metadata_payload(log: LogRecord) -> dict[str, Any]:
    return {
        'hole_id': log.log_id,
        'source_root': str(log.archive_path),
        'source_las_member': log.las_member,
        'source_xhd_member': log.xhd_member,
        'source_xrd_member': log.xrd_member,
        'classification': classify_log(log),
        'log_created': str(log.log_created) if log.log_created is not None else None,
        'view_type': log.view_type,
        'index_unit': log.index_unit,
        'well_header': log.well,
        'sonde_stack': log.xhd.get('selected_stack', []),
        'curves': [curve.__dict__ for curve in log.curves],
        'warnings': log_qc_issues(log),
        'caliper_interpretation': {
            'channel_meaning': 'tool_to_wall_range',
            'contact_coordinates': {
                'X1': '(+X1, 0)', 'X2': '(-X2, 0)',
                'Y1': '(0, +Y1)', 'Y2': '(0, -Y2)',
            },
            'opposite_sums_meaning': 'tool_axis_chords_not_diameters',
            'identifiability': 'one_parameter_exact_ellipse_family',
            'unconstrained_upper_area': 'unbounded',
            'calculation_profile': CALIPER_CALCULATION_PROFILE,
            'ellipse_constraints': _normalise_ellipse_constraints(CALIPER_ELLIPSE_CONSTRAINTS),
        },
    }


def build_caliper_app_table(geometry: pd.DataFrame, hole_id: str) -> pd.DataFrame:
    columns = [
        'depth_m', 'X1', 'X2', 'Y1', 'Y2', 'motor_current_ma',
        'x_chord_mm', 'y_chord_mm', 'maximum_observed_chord_mm',
        'contact_polygon_area_mm2', 'contact_polygon_area_equivalent_diameter_mm',
        'absolute_min_rho', 'absolute_min_area_mm2', 'absolute_min_area_equivalent_diameter_mm',
        'lower_rho', 'lower_area_mm2', 'lower_area_equivalent_diameter_mm',
        'lower_centre_x_mm', 'lower_centre_y_mm', 'lower_major_diameter_mm',
        'lower_minor_diameter_mm', 'lower_axis_ratio', 'lower_orientation_deg',
        'reference_rho', 'reference_area_mm2', 'reference_area_equivalent_diameter_mm',
        'reference_centre_x_mm', 'reference_centre_y_mm', 'reference_major_diameter_mm',
        'reference_minor_diameter_mm', 'reference_axis_ratio', 'reference_orientation_deg',
        'upper_bound_area_mm2', 'upper_bound_area_equivalent_diameter_mm',
        'upper_scenario_rho', 'upper_scenario_area_mm2', 'upper_scenario_area_equivalent_diameter_mm',
        'upper_scenario_centre_x_mm', 'upper_scenario_centre_y_mm', 'upper_scenario_major_diameter_mm',
        'upper_scenario_minor_diameter_mm', 'upper_scenario_axis_ratio', 'upper_scenario_orientation_deg',
        'ellipse_upper_bound_finite', 'ellipse_constraint_status',
        'ellipse_constraint_feasible_fraction', 'segment_integrable', 'delta_depth_m',
        'segment_volume_contact_polygon_lower_m3', 'cumulative_volume_contact_polygon_lower_m3',
        'segment_volume_ellipse_lower_m3', 'cumulative_volume_ellipse_lower_m3',
        'segment_volume_ellipse_reference_m3', 'cumulative_volume_ellipse_reference_m3',
        'segment_volume_ellipse_upper_bound_m3', 'cumulative_volume_ellipse_upper_bound_m3',
        'segment_volume_nominal_m3', 'cumulative_volume_nominal_m3',
    ]
    output = geometry[[column for column in columns if column in geometry]].copy()
    output.insert(0, 'hole_id', hole_id)
    output['calculation_profile'] = geometry.attrs.get('calculation_profile')
    output['identifiability'] = geometry.attrs.get('identifiability')
    output['ellipse_constraints'] = geometry.attrs.get('ellipse_constraints_label')
    return output


def safe_file_stem(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', value).strip('_') or 'wireline_log'


caliper_app_samples = build_caliper_app_table(caliper_geometry, calibrated_id)
metadata_payload = build_log_metadata_payload(calibrated_log)

display(caliper_app_samples.head())
display(pd.DataFrame([{
    'table': 'hole_log_samples',
    'grain': 'one row per sampled depth',
    'purpose': 'raw curves, direct chords, uncertainty products and segment-volume products',
}, {
    'table': 'hole_log_metadata',
    'grain': 'one row/document per log',
    'purpose': 'provenance, headers, sonde stack, calibration, interpretation profile and QC',
}, {
    'table': 'hole_plan',
    'grain': 'one row per planned hole',
    'purpose': 'planned depth, diameter, collar, trajectory and approved ellipse constraints',
}, {
    'table': 'hole_reconciliation',
    'grain': 'one row per inspected hole/run',
    'purpose': 'definite findings, indeterminate intervals, qualified volume bounds and review workflow',
}]))

DO_EXPORT = False
if DO_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    export_stem = safe_file_stem(calibrated_id)
    summary_table.to_csv(EXPORT_DIR / 'wireline_log_inventory.csv', index=False)
    archive_inventory.to_csv(EXPORT_DIR / 'folder_companion_inventory.csv', index=False)
    caliper_app_samples.to_csv(EXPORT_DIR / f'{export_stem}_caliper_uncertainty_samples.csv', index=False)
    register_reconciliation.to_csv(EXPORT_DIR / f'{export_stem}_reconciliation.csv', index=False)
    interval_summary.to_csv(EXPORT_DIR / f'{export_stem}_interval_summary.csv', index=False)
    with (EXPORT_DIR / f'{export_stem}_metadata.json').open('w', encoding='utf-8') as handle:
        json.dump(metadata_payload, handle, indent=2, default=str)
    if trajectory is not None:
        trajectory_stem = safe_file_stem(str(trajectory.attrs.get('log_id', 'trajectory')))
        trajectory.to_csv(EXPORT_DIR / f'{trajectory_stem}_trajectory.csv', index=False)
    print(f'Exports written to: {EXPORT_DIR.resolve()}')
else:
    print('Set DO_EXPORT = True to write CSV/JSON app hand-off files.')

## 15. Folder-data and uncertainty-model regression checks

In [ ]:
# Generic regression checks for the currently selected folder.
assert INPUT_FOLDER.exists(), f'Input folder not found: {INPUT_FOLDER}'
assert INPUT_FOLDER.is_dir(), f'Input path is not a folder: {INPUT_FOLDER}'
assert len(MEASUREMENT_FILES) >= 1, 'No supported measurement files were discovered.'
assert len(logs) >= 1, 'No LAS logs were loaded.'
assert not archive_inventory.empty, 'The companion-file inventory is empty.'

required_inventory_columns = {
    'source_folder', 'item', 'LAS', 'XHD', 'XRD', 'complete_triplet'
}
missing_inventory_columns = required_inventory_columns - set(archive_inventory.columns)
assert not missing_inventory_columns, (
    f'Inventory is missing columns: {sorted(missing_inventory_columns)}'
)

for log_id, log in logs.items():
    assert not log.df.empty, f'{log_id}: parsed dataframe is empty.'
    assert len(log.curves) == log.df.shape[1], (
        f'{log_id}: curve metadata does not match dataframe columns.'
    )
    depth_column = physical_depth_column(log)
    assert depth_column in log.df.columns, (
        f'{log_id}: index column {depth_column!r} is missing.'
    )
    values = pd.to_numeric(log.df[depth_column], errors='coerce').dropna()
    assert len(values) >= 1, f'{log_id}: index column has no numeric values.'

assert len(caliper_ids) >= 1, 'No four-arm caliper log was detected.'
assert all(calibrated_log.curve(arm) is not None for arm in ARM_NAMES)
assert (caliper_geometry[list(ARM_NAMES)] > 0).all().all()
assert (caliper_geometry['x_chord_mm'] == caliper_geometry['X1'] + caliper_geometry['X2']).all()
assert (caliper_geometry['y_chord_mm'] == caliper_geometry['Y1'] + caliper_geometry['Y2']).all()
assert caliper_geometry['segment_volume_ellipse_lower_m3'].sum() > 0
assert caliper_geometry['segment_volume_ellipse_reference_m3'].sum() > 0
assert caliper_geometry['lower_area_equivalent_diameter_mm'].notna().all()
assert caliper_geometry['reference_area_equivalent_diameter_mm'].notna().all()
assert (
    caliper_geometry['reference_area_mm2'] + 1e-6
    >= caliper_geometry['absolute_min_area_mm2']
).all()
assert caliper_summary['integrated_length_m'].iloc[0] > 0
assert caliper_summary['identifiability'].iloc[0] == 'UNDERDETERMINED'
finite_upper_rows = caliper_geometry['ellipse_upper_bound_finite']
if finite_upper_rows.any():
    assert (
        caliper_geometry.loc[finite_upper_rows, 'upper_bound_area_mm2']
        + 1e-6
        >= caliper_geometry.loc[finite_upper_rows, 'lower_area_mm2']
    ).all()
    scenario_rows = finite_upper_rows & caliper_geometry['upper_scenario_area_mm2'].notna()
    assert (
        caliper_geometry.loc[scenario_rows, 'upper_bound_area_mm2']
        + 1e-6
        >= caliper_geometry.loc[scenario_rows, 'upper_scenario_area_mm2']
    ).all()

# Verify that selected ellipse models pass through all four measured contacts.
validation_row = caliper_geometry.iloc[len(caliper_geometry) // 2]
validation_ranges = tuple(float(validation_row[name]) for name in ARM_NAMES)
for prefix in ('absolute_min', 'lower', 'reference'):
    parameters = ellipse_parameters_from_ranges(*validation_ranges, float(validation_row[f'{prefix}_rho']))
    residuals = ellipse_contact_residuals_mm(parameters, *validation_ranges)
    assert np.max(np.abs(residuals)) < 1e-6, (
        f'{prefix} ellipse does not reproduce the four contacts: {residuals}'
    )

constraints = _normalise_ellipse_constraints(CALIPER_ELLIPSE_CONSTRAINTS)
if constraints['max_axis_ratio'] is None and constraints['max_major_diameter_mm'] is None:
    assert not caliper_summary['ellipse_upper_bound_finite_for_full_interval'].iloc[0]
    assert caliper_summary['ellipse_model_volume_upper_m3'].isna().all()

# Calibration assertions are applied only when calibration records exist.
if not cal_table.empty:
    assert set(cal_table['channel']).issubset(set(ARM_NAMES))
    assert cal_table['mode'].notna().all()

if trajectory is not None:
    assert trajectory['tvd_m'].iloc[-1] >= 0
    assert np.isfinite(
        trajectory[['easting_offset_m', 'northing_offset_m', 'tvd_m']].to_numpy()
    ).all()

incomplete_items = archive_inventory.loc[
    ~archive_inventory['complete_triplet']
].copy()

print(
    f'All parser, metadata, contact-geometry, uncertainty, calibration and trajectory checks passed '
    f'for {len(logs)} LAS logs.'
)
if not incomplete_items.empty:
    print(
        f'{len(incomplete_items)} item(s) do not contain a complete '
        'LAS/XHD/XRD companion set.'
    )

## 16. Conclusions and path to the MiningSim web application

### What changed in the caliper interpretation

- The caliper channels are four tool-to-wall ranges from a generally off-centre tool reference.
- `X1 + X2` and `Y1 + Y2` are retained as observed chord lengths, but are no longer labelled or used as diameters.
- The previous exact `diameter_x`, `diameter_y`, `ovality`, `hole centre`, `tool offset` and `measured volume` products have been removed.
- Four contacts define a one-parameter family of exact ellipses. The notebook visualises that family and stores the free parameter `ρ` with each selected scenario.
- The unconstrained ellipse family has a finite minimum area but no finite maximum area. Therefore the notebook reports an ellipse-model lower volume, an axis-aligned reference scenario and, only where explicit priors permit it, a constrained upper volume.
- The direct contact polygon is exported separately. Its area is a convexity-qualified lower bound, not an ellipse estimate.
- Reconciliation now distinguishes definite oversize, prior-qualified pass conditions and indeterminate intervals. It does not produce an unconditional pass from four contacts alone.

### Implications for existing results

Any earlier outputs based on adding opposite readings as diameters should be treated as superseded. This includes previously calculated equivalent diameter, ovality, inferred hole centre, tool eccentricity, cross-sectional area, cumulative volume and pass/fail status. Raw curves, calibration checks, file parsing, trajectory calculations and non-caliper visualisations remain valid.

### Recommended application decomposition

1. **Ingestion service** — folder/LAS/XHD validation, checksum, provenance and immutable raw-file storage.
2. **Canonical log model** — hole/run metadata, sonde-owned curves, units, depth basis and QC flags.
3. **Observation layer** — four ranges, contact coordinates, tool-axis chords and contact-polygon metrics.
4. **Inference layer** — versioned ellipse-family calculations, explicit constraint profiles, scenario provenance and uncertainty bounds.
5. **Inspection interface** — interactive tracks, feasible cross-section families, direct 3D contact traces, interval exceptions and approval workflow.
6. **Persistence/API** — separate raw samples, observed products, model-derived products, plan records and reconciliation decisions so every result is reproducible and auditable.

### Information required to reduce uncertainty

A finite and operationally useful volume interval requires independent information. Suitable sources may include a documented maximum axis ratio or breakout envelope, a maximum physically possible borehole diameter, centraliser/tool-body geometry, inclination-dependent tool-position constraints, oriented multi-arm measurements, repeat runs at materially different tool positions, acoustic/optical imaging, or another independent diameter measurement. Constraints must be stored with their source, approval status and calculation-profile version.